## Ajustement 61010 à 15 min + réallocation (Run all) + exports Nexus

### Objectif
- **Limiter chaque prestation 61010 à 15 minutes**
- **Réallouer le surplus uniquement** vers les prestations autorisées :
  `11000, 11100, 11200, 14000, 14100, 14200`
- Générer un fichier **RDA ajusté** + des **exports Nexus** (CSV + .bat) + des **contrôles** (différences, overlaps, dashboard)

---

## Fichiers nécessaires (déjà dans Google Drive)
- **RDA brut** (format `.xlsx` ou `.csv`) contenant au minimum :
  - `Date` / `Date Début` / `Jour`
  - `Début` et `Fin`
  - `No prestation` (ou équivalent)
  - `Durée`
  - `No collaborateur`
  - `N° du client`
- (Optionnel) Un nom de dossier d’export (sinon il est auto-déduit du mois trouvé dans le fichier)

---

## Exécution (Run all)
1. Ouvrez le notebook dans Google Colab.
2. Dans la cellule **0 — Setup**, renseignez / vérifiez :
   - `entite_oe` (NE 301 / SARL 201 / SA 101) → définit l’**OE** pour les exports
   - `ROOT_PARENT` (dossier Drive où créer les exports)
   - `EXPORT_ROOT_NAME` (nom du dossier d’export, optionnel)
   - `APPLY_15MIN_ADJUSTMENT = True` (pour activer l’ajustement)
   - **IMPORTANT** : mettez le chemin du fichier dans `EXCEL_OR_CSV_FILE` (dans Drive)  
     *(dans votre version actuelle il est `" "` → à remplacer par un chemin valide)*
3. Menu : **Runtime / Exécution → Run all / Tout exécuter**.
4. Autorisez l’accès à Google Drive si demandé, puis laissez le script aller jusqu’aux ✅.

---

## Contrôle obligatoire : vérifier que l’allocation est conforme (STRICT GUARDS)
Pendant l’exécution de la cellule d’ajustement, le script affiche :

- `Sum delta (must be 0): 0`  → la somme des minutes ajoutées / retirées est neutre
- `Receiving on non-allowed codes: 0` → **aucun code hors whitelist** ne reçoit des minutes
- `Giving on non-61010 codes: 0` → **seul 61010** perd des minutes

### Si l’un de ces points n’est pas à 0
Le script **STOP** avec une erreur (c’est normal) : il faut corriger les données / colonnes / formats avant de relancer.

---

## Où trouver les fichiers générés
Dans Google Drive, sous :
- `ROOT_PARENT / (EXPORT_ROOT_NAME ou RDA-MMYYYY)`

Vous trouverez notamment :
- Le fichier **RDA ajusté** : `*_61010_adjusted_no_overlap.*` (xlsx/csv)
- Le fichier **Différences RAW vs ADJ** : `RDA_StartEnd_Differences_RAW_vs_ADJ.xlsx`
- Le rapport **Durée par collaborateur** : `RDA_duree_check.csv`
- Les exports Nexus :
  - `01_All_Collabs_One_CSV/` (1 CSV + 1 .bat)
  - `02_Collabs_With_61010_One_CSV/` (1 CSV + 1 .bat si des 61010 existent)
  - `03_Per_Collab_Separate/` (1 dossier par collaborateur, avec CSV + .bat)
- La table des codes prestations :
  - `HAS_map.csv`
- Le dashboard final :
  - `Collaborator_Client_Summary.xlsx` (collaborateurs + clients extraits)

---

## Contrôle recommandé : overlaps après ajustement
Le script exécute un check “overlap” (même collaborateur + même jour).
- Résultat attendu : `✅ No overlaps detected ...`
- Si des overlaps existent : un tableau s’affiche avec les paires en conflit (à analyser avant import).

In [ ]:
#@title 0 — Setup (Drive, configuration & upload) { form-width: "600px" }

# 1) Monte Google Drive
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)

# 2) Imports communs
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# 3) Paramètres de base (sélection dans Colab)
#    - Entité / UO / OE
entite_oe = "NE 301"  #@param ["NE 301", "SARL 201", "SA 101"]
#    - Dossier racine où seront créés les exports (dans Drive)
ROOT_PARENT = "/content/drive/MyDrive"  #@param {type:"string"}
#    - Nom du dossier d’export (facultatif : si vide, on utilisera RDA-MMYYYY)
EXPORT_ROOT_NAME = "NE_15minsreduit_April2026"  #@param {type:"string"}

#    - Appliquer l’ajustement 61010 à 15 minutes ?
APPLY_15MIN_ADJUSTMENT = True  #@param {type:"boolean"}

#    - Activer le transfert des prestations whitelistées (201 -> 101) ?
ENABLE_WHITELIST_TRANSFER = False  #@param {type:"boolean"}

#    - Prestation whitelist diverted to the 201 -> 101 transfer path
PRESTATION_WHITELIST_STR = "16011,909,16009,195"  #@param {type:"string"}
TRANSFER_TARGET_OE_VALUE = "100000000000000101"  # SA 101, fixed for transfer
MAPPING_XLSX_FILE = " "  #@param {type:"string"}

# Known mapping files to try automatically when MAPPING_XLSX_FILE is blank.
MATCH_CANDIDATES = [
    "/content/matched_clients_and_collabs_20251118_014432.xlsx",
    "/content/matched_clients_and_collabs_2025sept.xlsx",
]

# Optional mapping-column overrides. Leave blank for automatic detection.
SOURCE_CLIENT_MAP_COL = ""  #@param {type:"string"}
TARGET_CLIENT_MAP_COL = ""  #@param {type:"string"}
SOURCE_COLLAB_MAP_COL = ""  #@param {type:"string"}
TARGET_COLLAB_MAP_COL = ""  #@param {type:"string"}

# Mapping entité -> OE complet
OE_MAP = {
    "NE 301":  "100000000000000301",
    "SARL 201": "100000000000000201",
    "SA 101":  "100000000000000101",
}
OE_VALUE = OE_MAP[entite_oe]

def _norm_setup_code(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    m = re.search(r"\d+", s)
    return m.group(0) if m else s

PRESTATION_WHITELIST = {
    _norm_setup_code(x)
    for x in PRESTATION_WHITELIST_STR.split(",")
    if _norm_setup_code(x)
} if ENABLE_WHITELIST_TRANSFER else set()

print(f"Entité sélectionnée : {entite_oe}  →  OE = {OE_VALUE}")
print(f"Réduction 61010 à 15 minutes activée : {APPLY_15MIN_ADJUSTMENT}")
print(f"Transfert whitelist (101) activé : {ENABLE_WHITELIST_TRANSFER}")
print(f"Prestations transférées vers 101 : {sorted(PRESTATION_WHITELIST)}")
print("➡️ Sélectionne ton fichier RDA brut (Excel ou CSV) à importer…")

# Manually set the input file path to bypass interactive upload
# Please change this path if you want to use a different file, or delete it
# to trigger interactive upload if the file is not found.
EXCEL_OR_CSV_FILE = " "
input_path = Path(EXCEL_OR_CSV_FILE)

if not input_path.exists():
    print(f"Input file not found at {input_path}. Trying interactive upload...")
    uploaded = files.upload()
    if uploaded:
        # Get the name of the first uploaded file
        EXCEL_OR_CSV_FILE = next(iter(uploaded.keys()))
        input_path = Path(EXCEL_OR_CSV_FILE)
        print(f"Uploaded file: {input_path.name}")
    else:
        raise FileNotFoundError(f"No file uploaded and '{EXCEL_OR_CSV_FILE}' does not exist. Please upload the file or provide a valid path in EXCEL_OR_CSV_FILE.")

print(f"📂 Fichier chargé : {input_path.name}")
print(f"Les exports seront créés sous : {ROOT_PARENT}")
if EXPORT_ROOT_NAME.strip():
    print(f"Dossier d’export demandé : {EXPORT_ROOT_NAME.strip()}")

# Mapping workbook is resolved during setup, right after the raw RDA file.
MAPPING_XLSX_PATH = None

def resolve_mapping_workbook():
    if not globals().get("ENABLE_WHITELIST_TRANSFER", True):
        return None

    global MAPPING_XLSX_PATH
    if MAPPING_XLSX_PATH:
        p = Path(str(MAPPING_XLSX_PATH))
        if p.exists():
            return str(p)
    configured = str(globals().get("MAPPING_XLSX_FILE", "")).strip()
    if configured:
        p = Path(configured)
        if p.exists():
            MAPPING_XLSX_PATH = str(p)
            print(f"Using configured mapping workbook: {MAPPING_XLSX_PATH}")
            return MAPPING_XLSX_PATH
        print(f"Configured mapping workbook not found: {p}")

    for candidate in globals().get("MATCH_CANDIDATES", []):
        p = Path(candidate)
        if p.exists():
            MAPPING_XLSX_PATH = str(p)
            print(f"Using known mapping workbook: {MAPPING_XLSX_PATH}")
            return MAPPING_XLSX_PATH

    print("Upload the 201 -> 101 mapping workbook (Excel with clients and collabs tabs).")
    uploaded_map = files.upload()
    if not uploaded_map:
        raise FileNotFoundError("No mapping workbook uploaded for the whitelisted 201 -> 101 transfer.")
    map_name = next(iter(uploaded_map.keys()))
    candidates = [Path(map_name), Path('/content') / map_name]
    selected = next((p for p in candidates if p.exists()), candidates[0])
    MAPPING_XLSX_PATH = str(selected)
    print(f"Uploaded mapping workbook: {MAPPING_XLSX_PATH}")
    return MAPPING_XLSX_PATH

print("\nMapping workbook for 201 -> 101 transfer:")
MAPPING_XLSX_PATH = resolve_mapping_workbook()
if MAPPING_XLSX_PATH:
    print(f"Mapping workbook ready: {MAPPING_XLSX_PATH}")
else:
    print("Mapping workbook skipped (whitelist transfer disabled).")


Mounted at /content/drive
Entité sélectionnée : NE 301  →  OE = 100000000000000301
Réduction 61010 à 15 minutes activée : True
Transfert whitelist (101) activé : False
Prestations transférées vers 101 : []
➡️ Sélectionne ton fichier RDA brut (Excel ou CSV) à importer…
Input file not found at  . Trying interactive upload...


Saving ne_april.xlsx to ne_april (3).xlsx
Uploaded file: ne_april (3).xlsx
📂 Fichier chargé : ne_april (3).xlsx
Les exports seront créés sous : /content/drive/MyDrive
Dossier d’export demandé : NE_15minsreduit_April2026

Mapping workbook for 201 -> 101 transfer:
Mapping workbook skipped (whitelist transfer disabled).


In [ ]:
# @title
# === Cell 1 — Load RAW RDA into df & show basic info ===

input_path = Path(EXCEL_OR_CSV_FILE)
if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {input_path}")

print(f"Loading RAW RDA from: {input_path}")

ext = input_path.suffix.lower()
if ext == ".xlsx":
    df = pd.read_excel(input_path)
elif ext == ".csv":
    df = pd.read_csv(input_path)
else:
    raise ValueError(f"Unsupported file type: {ext} (use .xlsx or .csv)")

print("\nColumns found in RAW RDA:")
print(df.columns.tolist())
display(df.head(10))

# OUTPUT_FILE : adjusted version in Drive

# Determine root_folder for all exports early
DATE_COL_CANDIDATES    = ['Date Début', 'Date', 'Jour', 'Date de prestation']
date_col = next((c for c in DATE_COL_CANDIDATES if c in df.columns), None);
if date_col is None:
    raise ValueError(f"Could not find a date column for determining export folder; tried {DATE_COL_CANDIDATES}")

date_series = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
first_valid = date_series.dropna().iloc[0]
month = int(first_valid.month)
year  = int(first_valid.year)

export_root_name = EXPORT_ROOT_NAME.strip()
if export_root_name:
    root_folder_name = export_root_name
else:
    root_folder_name = f"RDA-{month:02d}{year}"

root_folder = Path(ROOT_PARENT) / root_folder_name
root_folder.mkdir(parents=True, exist_ok=True)

# Define OUTPUT_FILE to be within the determined root_folder
OUTPUT_FILE = root_folder / f"{input_path.stem}_61010_adjusted_no_overlap{input_path.suffix}"
print(f"\nAdjusted file will be saved to:\n  {OUTPUT_FILE}")

Loading RAW RDA from: ne_april (3).xlsx

Columns found in RAW RDA:
['Statut', 'Facturation', 'Jour', 'No collaborateur', 'Collaborateur', 'Durée', 'Début', 'Fin', 'N° du client', 'Client', 'No prestation', 'Prestation', 'UO', 'Date de création', "Manière d'enregistrement"]


,Statut,Facturation,Jour,No collaborateur,Collaborateur,Durée,Début,Fin,N° du client,Client,No prestation,Prestation,UO,Date de création,Manière d'enregistrement
0,OK,Non validé,01.04.2026,1209,Sihyürek Mine,35,01.04.2026 10:08,01.04.2026 10:43,NaN,,61010,Temps de déplacement soins,Home Assistance (Neuchatel) SA,01.04.2026 10:43:15,asebis smart
1,OK,Non validé,01.04.2026,1209,Sihyürek Mine,22,01.04.2026 18:25,01.04.2026 18:47,202863.0,LEUBA Erica,11100,OPAS-B Examens et traitements,Home Assistance (Neuchatel) SA,01.04.2026 18:52:46,asebis smart
2,OK,Non validé,01.04.2026,1209,Sihyürek Mine,26,01.04.2026 09:06,01.04.2026 09:32,NaN,,61010,Temps de déplacement soins,Home Assistance (Neuchatel) SA,01.04.2026 09:30:39,asebis smart
3,OK,Non validé,13.04.2026,1209,Sihyürek Mine,20,13.04.2026 11:27,13.04.2026 11:47,104115.0,BOSCOE EDITH,11100,OPAS-B Examens et traitements,Home Assistance (Neuchatel) SA,13.04.2026 12:48:35,asebis smart
4,OK,Non validé,21.04.2026,1209,Sihyürek Mine,21,21.04.2026 18:40,21.04.2026 19:01,NaN,,61010,Temps de déplacement soins,Home Assistance (Neuchatel) SA,21.04.2026 19:01:47,asebis smart
5,OK,Non validé,01.04.2026,1209,Sihyürek Mine,15,01.04.2026 10:43,01.04.2026 10:58,202896.0,JEAN-MAIRET Patrice,11100,OPAS-B Examens et traitements,Home Assistance (Neuchatel) SA,01.04.2026 11:03:34,asebis smart
6,OK,Non validé,01.04.2026,1209,Sihyürek Mine,28,01.04.2026 15:55,01.04.2026 16:23,NaN,,61010,Temps de déplacement soins,Home Assistance (Neuchatel) SA,01.04.2026 16:28:36,asebis smart
7,OK,Non validé,01.04.2026,1209,Sihyürek Mine,20,01.04.2026 08:26,01.04.2026 08:46,NaN,,61010,Temps de déplacement soins,Home Assistance (Neuchatel) SA,01.04.2026 08:52:47,asebis smart
8,OK,Non validé,01.04.2026,1209,Sihyürek Mine,25,01.04.2026 07:19,01.04.2026 07:44,202940.0,MILAZZO Filippo,11100,OPAS-B Examens et traitements,Home Assistance (Neuchatel) SA,01.04.2026 07:46:42,asebis smart
9,OK,Non validé,01.04.2026,1209,Sihyürek Mine,12,01.04.2026 08:14,01.04.2026 08:26,103830.0,MAS Marie,11200,OPAS-C Soins de base,Home Assistance (Neuchatel) SA,01.04.2026 08:47:28,asebis smart



Adjusted file will be saved to:
  /content/drive/MyDrive/NE_15minsreduit_April2026/ne_april (3)_61010_adjusted_no_overlap.xlsx


In [ ]:
# @title Cell 1B — Sanity check Durée vs Début/Fin + (option) normaliser Durée

import datetime as dt
import numpy as np
import pandas as pd
from IPython.display import display

# ---- Re-detect cols (same logic as Cell 2) ----
TIME_COL_START = 'Début'
TIME_COL_END   = 'Fin'
FORCE_DUREE_FROM_TIMES = False


DATE_COL_CANDIDATES    = ['Date Début', 'Date', 'Jour', 'Date de prestation']
COLLAB_COL_CANDIDATES  = ['No collaborateur', 'Collaborateur', 'ID collaborateur']
DUREE_COL_CANDIDATES   = ['Durée', 'Duree', 'Durée (min)', 'Durée (minutes)']
CODE_COL_CANDIDATES    = ['No prestation', 'Code prestation', 'Prestation', 'Code', 'N° Prestation']

code_col  = next((c for c in CODE_COL_CANDIDATES if c in df.columns), None)
date_col  = next((c for c in DATE_COL_CANDIDATES if c in df.columns), None)
collab_col= next((c for c in COLLAB_COL_CANDIDATES if c in df.columns), None)
duree_col = next((c for c in DUREE_COL_CANDIDATES if c in df.columns), None)

if code_col is None or date_col is None or collab_col is None or duree_col is None:
    raise ValueError(f"Colonnes manquantes. Trouvé: code={code_col}, date={date_col}, collab={collab_col}, durée={duree_col}")

print("Sanity check colonnes:")
print("  code_col :", code_col)
print("  date_col :", date_col)
print("  collab_col:", collab_col)
print("  duree_col:", duree_col)

# ---- Robust time parse ----
def to_min_robust(t):
    if pd.isna(t):
        return np.nan

    # pandas Timestamp
    if isinstance(t, pd.Timestamp):
        return int(t.hour) * 60 + int(t.minute)

    # datetime.time
    if isinstance(t, dt.time):
        return int(t.hour) * 60 + int(t.minute)

    # Excel float time (0..1)
    if isinstance(t, (float, int)) and not isinstance(t, bool):
        x = float(t)
        if 0 <= x < 1.0:
            return int(round(x * 24 * 60)) % (24 * 60)
        # sometimes minutes since midnight
        if 0 <= x < 24 * 60 and abs(x - round(x)) < 1e-9:
            return int(round(x))
        # sometimes hours as decimal
        if 0 <= x < 24:
            return int(round(x * 60))

    # string parse
    s = str(t).strip()
    # try datetime parse (keeps hour/min)
    dtv = pd.to_datetime(s, dayfirst=True, errors='coerce')
    if not pd.isna(dtv):
        return int(dtv.hour) * 60 + int(dtv.minute)

    # fallback HH:MM[:SS]
    tok = s.split()[-1]
    parts = tok.split(':')
    if len(parts) >= 2 and parts[0].isdigit() and parts[1].isdigit():
        return int(parts[0]) * 60 + int(parts[1])

    raise ValueError(f"Format heure inattendu: {t!r}")

def diff_with_midnight_wrap(end_min, start_min):
    if pd.isna(end_min) or pd.isna(start_min):
        return np.nan
    d = int(end_min) - int(start_min)
    return d if d >= 0 else d + 24 * 60

# ---- Compute calc duration ----
tmp = df.copy()
tmp['_start_min_calc'] = tmp[TIME_COL_START].apply(to_min_robust)
tmp['_end_min_calc']   = tmp[TIME_COL_END].apply(to_min_robust)
tmp['_dur_calc']       = tmp.apply(lambda r: diff_with_midnight_wrap(r['_end_min_calc'], r['_start_min_calc']), axis=1).astype('Int64')

tmp['_duree_col']      = pd.to_numeric(tmp[duree_col], errors='coerce').astype('Int64')
tmp['_delta']          = (tmp['_duree_col'] - tmp['_dur_calc']).astype('Int64')

total_col  = int(tmp['_duree_col'].fillna(0).sum())
total_calc = int(tmp['_dur_calc'].fillna(0).sum())
print("\nTotaux BRUT:")
print("  Durée (colonne)  :", total_col)
print("  Durée (calculée) :", total_calc)
print("  Diff (col - calc):", total_col - total_calc)

bad = tmp[tmp['_delta'].fillna(0) != 0].copy()
print(f"\nLignes incohérentes (Durée != Fin-Début): {len(bad)} / {len(tmp)}")
if len(bad) > 0:
    display(
        bad[[date_col, collab_col, code_col, TIME_COL_START, TIME_COL_END, duree_col, '_dur_calc', '_delta']]
        .head(50)
    )
    print("\nTop causes par code prestation (somme delta):")
    display(
        bad.groupby(code_col)['_delta'].sum().sort_values(ascending=False).head(20).reset_index()
    )

# ---- OPTION: normalize Durée from times (recommended for clean exports/dashboard) ----
FORCE_DUREE_FROM_TIMES = True  # <-- mets False si tu veux juste diagnostiquer

if FORCE_DUREE_FROM_TIMES:
    if f"{duree_col}_original" not in df.columns:
        df[f"{duree_col}_original"] = df[duree_col]
    # keep original if calc is NaN
    df[duree_col] = tmp['_dur_calc'].where(tmp['_dur_calc'].notna(), tmp['_duree_col']).astype('Int64')
    print(f"\n✅ Durée normalisée depuis Début/Fin dans df[{duree_col!r}] (backup: {duree_col}_original)")


Sanity check colonnes:
  code_col : No prestation
  date_col : Jour
  collab_col: No collaborateur
  duree_col: Durée

Totaux BRUT:
  Durée (colonne)  : 60216
  Durée (calculée) : 60216
  Diff (col - calc): 0

Lignes incohérentes (Durée != Fin-Début): 0 / 3120

✅ Durée normalisée depuis Début/Fin dans df['Durée'] (backup: Durée_original)


In [ ]:
#@title ✅ Cell 2 (STRICT) — 61010 cap 15 + surplus ONLY to allowed codes, with HARD DELTA GUARDS

try:
    df
    OUTPUT_FILE
except NameError:
    raise NameError("Run Cell 1 first so 'df' and 'OUTPUT_FILE' exist.")

import pandas as pd
import numpy as np
from pathlib import Path
import datetime as dt
import re
from IPython.display import display

ALLOWED_TARGET_CODES = {"11000", "11100", "11200", "14000", "14100", "14200"}
CODE_61010 = "61010"

TIME_COL_START = "Début"
TIME_COL_END   = "Fin"

DATE_COL_CANDIDATES    = ["Date Début", "Date", "Jour", "Date de prestation"]
COLLAB_COL_CANDIDATES  = ["No collaborateur", "Collaborateur", "ID collaborateur"]
DUREE_COL_CANDIDATES   = ["Durée", "Duree", "Durée (min)", "Durée (minutes)"]
CODE_COL_CANDIDATES    = ["N° Prestation", "No prestation", "Code prestation", "Prestation", "Code", "N° Prestation"]

PRESERVE_DAY_START_END = True

for col in (TIME_COL_START, TIME_COL_END):
    if col not in df.columns:
        raise KeyError(f"Missing required time column: {col!r}")

code_col = next((c for c in CODE_COL_CANDIDATES if c in df.columns), None)
date_col = next((c for c in DATE_COL_CANDIDATES if c in df.columns), None)
collab_col = next((c for c in COLLAB_COL_CANDIDATES if c in df.columns), None)
duree_col = next((c for c in DUREE_COL_CANDIDATES if c in df.columns), None)

if code_col is None or date_col is None or collab_col is None:
    raise ValueError(f"Missing columns. Found: code={code_col}, date={date_col}, collab={collab_col}")

print(f"Using prestation column : {code_col}")
print(f"Using date column       : {date_col}")
print(f"Using collab column     : {collab_col}")
print(f"Using durée column      : {duree_col}")

def norm_code(x) -> str:
    """
    More robust normalization:
    - trims
    - turns 11000.0 -> 11000
    - if string contains digits + extra text, extract the LONGEST digit run
    """
    if pd.isna(x):
        return ""
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, (float, np.floating)):
        if np.isfinite(x) and abs(x - round(x)) < 1e-9:
            return str(int(round(x)))
        s = str(x).strip()
    else:
        s = str(x).strip()

    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]

    if s.isdigit():
        return s

    runs = re.findall(r"\d+", s)
    if runs:
        runs = sorted(runs, key=len, reverse=True)
        return runs[0]
    return s

ALLOWED_TARGET_CODES_NORM = {norm_code(c) for c in ALLOWED_TARGET_CODES}
CODE_61010_NORM = norm_code(CODE_61010)

df[code_col] = df[code_col].apply(norm_code)

# -------------------- WHITELIST FORK: remove 201 -> 101 rows from the main reduction path --------------------
PRESTATION_WHITELIST_NORM = {
    norm_code(c)
    for c in globals().get("PRESTATION_WHITELIST", {"16011", "909", "16009", "195"})
}

_whitelist_mask = df[code_col].apply(norm_code).isin(PRESTATION_WHITELIST_NORM)
df_whitelist = df.loc[_whitelist_mask].copy()
df_main_raw = df.loc[~_whitelist_mask].copy()
df_reduction_input = df_main_raw.copy()

print("\n=== WHITELIST FORK ===")
print("Whitelisted prestation codes diverted to 101:", sorted(PRESTATION_WHITELIST_NORM))
print("Rows diverted to 101 transfer path:", len(df_whitelist))
print("Rows kept in main 15-minute path:", len(df_reduction_input))

if len(df_whitelist) > 0:
    display(df_whitelist[[c for c in [date_col, collab_col, code_col, duree_col] if c in df_whitelist.columns]].head(20))

def to_min(t):
    if pd.isna(t):
        return np.nan
    if isinstance(t, pd.Timestamp):
        return int(t.hour) * 60 + int(t.minute)
    if isinstance(t, dt.time):
        return int(t.hour) * 60 + int(t.minute)
    if isinstance(t, (float, int)) and not isinstance(t, bool):
        x = float(t)
        if 0 <= x < 1.0:
            return int(round(x * 24 * 60)) % (24 * 60)
        if 0 <= x < 24 * 60 and abs(x - round(x)) < 1e-9:
            return int(round(x))
        if 0 <= x < 24:
            return int(round(x * 60))
    s = str(t).strip()
    dtv = pd.to_datetime(s, dayfirst=True, errors="coerce")
    if not pd.isna(dtv):
        return int(dtv.hour) * 60 + int(dtv.minute)
    tok = s.split()[-1]
    parts = tok.split(":")
    if len(parts) >= 2 and parts[0].isdigit() and parts[1].isdigit():
        return int(parts[0]) * 60 + int(parts[1])
    return np.nan

def diff_with_midnight_wrap(end_min, start_min):
    if pd.isna(end_min) or pd.isna(start_min):
        return np.nan
    d = int(end_min) - int(start_min)
    return d if d >= 0 else d + 24 * 60

def end_abs_min(end_min, start_min):
    if pd.isna(end_min) or pd.isna(start_min):
        return np.nan
    e = int(end_min)
    s = int(start_min)
    return e if e >= s else e + 1440

def minutes_to_hhmmss_mod(m_abs: int) -> str:
    if pd.isna(m_abs):
        return np.nan
    m = int(m_abs) % (24 * 60)
    h = m // 60
    mm = m % 60
    return f"{h:02d}:{mm:02d}:00"

original_cols = df_reduction_input.columns.tolist()
APPLY_15MIN_ADJUSTMENT = globals().get("APPLY_15MIN_ADJUSTMENT", True)

if df_reduction_input.empty:
    print("No non-whitelisted rows found; main 15-minute output will be empty.")
    df_out = df_reduction_input.copy()
    for extra in ["Temps retir\u00e9", "Temps ajout\u00e9"]:
        if extra not in df_out.columns:
            df_out[extra] = pd.Series(dtype="int64")
elif not APPLY_15MIN_ADJUSTMENT:
    print("15-min adjustment disabled => exporting non-whitelisted rows unchanged.")
    df_out = df_reduction_input.copy()
else:
    df_work = df_reduction_input.copy()

    # IMPORTANT: ALWAYS reset tracking cols (fixes 'old run' pollution)
    df_work["Temps retiré"] = 0
    df_work["Temps ajouté"] = 0

    df_work["_start_min"] = df_work[TIME_COL_START].apply(to_min)
    df_work["_end_min"]   = df_work[TIME_COL_END].apply(to_min)
    df_work["_start_abs0"]= df_work["_start_min"].astype("Float64")
    df_work["_end_abs0"]  = df_work.apply(lambda r: end_abs_min(r["_end_min"], r["_start_min"]), axis=1).astype("Float64")
    df_work["_dur0"]      = df_work.apply(lambda r: diff_with_midnight_wrap(r["_end_min"], r["_start_min"]), axis=1).astype("Int64")

    dts = pd.to_datetime(df_work[date_col], dayfirst=True, errors="coerce")
    df_work["_jour"] = dts.dt.date

    def is_allowed_target(code_str: str) -> bool:
        return norm_code(code_str) in ALLOWED_TARGET_CODES_NORM

    def process_block(block: pd.DataFrame) -> pd.DataFrame:
        b = block.sort_values(by=["_start_abs0", "_end_abs0"]).copy()
        b = b.reset_index(drop=False)
        idx_colname = "index"

        valid = b["_start_abs0"].notna() & b["_end_abs0"].notna() & b["_dur0"].notna()
        valid_idx = b.index[valid].tolist()
        if not valid_idx:
            return b.set_index(idx_colname).sort_index()

        # Save ORIGINAL durations for delta-based guards
        dur0 = pd.to_numeric(b["_dur0"], errors="coerce").fillna(0).astype(int)

        day_start = int(np.nanmin(b.loc[valid_idx, "_start_abs0"].astype(float)))
        day_end   = int(np.nanmax(b.loc[valid_idx, "_end_abs0"].astype(float)))
        day_span  = day_end - day_start

        order = b.loc[valid_idx].sort_values(by=["_start_abs0", "_end_abs0"]).index.tolist()

        gaps0 = []
        for k in range(len(order) - 1):
            i = order[k]
            j = order[k + 1]
            gi = max(0, int(b.loc[j, "_start_abs0"]) - int(b.loc[i, "_end_abs0"]))
            gaps0.append(gi)
        total_gap0 = sum(gaps0)

        dur_new = b["_dur0"].copy()
        target_idx = [i for i in valid_idx if is_allowed_target(b.loc[i, code_col])]

        def pick_nearest_target(i):
            if not target_idx:
                return None
            si = float(b.loc[i, "_start_abs0"])
            best = None
            best_dist = float('inf')
            for j in target_idx:
                sj = float(b.loc[j, "_start_abs0"])
                if sj >= si:  # Only consider targets that start at or after the 61010
                    dist = sj - si
                    if dist < best_dist:
                        best_dist = dist
                        best = j
            return best

        # --- Reallocation: ONLY 61010 gives, ONLY allowed targets receive ---
        for i in valid_idx:
            if norm_code(b.loc[i, code_col]) != CODE_61010_NORM:
                continue
            di = dur_new.loc[i]
            if pd.isna(di):
                continue
            di = int(di)
            if di <= 15:
                continue

            j = pick_nearest_target(i)
            if j is None:
                continue  # no allowed target in this (jour, collab)

            surplus = di - 15
            dur_new.loc[i] = 15
            dur_new.loc[j] = int(dur_new.loc[j]) + surplus

        # --- Build Temps ajouté/retiré purely from duration delta (no pollution possible) ---
        dn = pd.to_numeric(dur_new, errors="coerce").fillna(0).astype(int)
        delta = dn - dur0
        b["Temps ajouté"] = np.where(delta > 0, delta, 0).astype(int)
        b["Temps retiré"] = np.where(delta < 0, -delta, 0).astype(int)

        # --- Rebuild timeline (this can change Début/Fin for many rows; durations stay same except 61010 + targets) ---
        if PRESERVE_DAY_START_END:
            work_new = int(dn.loc[valid_idx].sum())
            slack = day_span - work_new

            if slack < 0:
                # fallback forward pack
                b["_start_abs"] = b["_start_abs0"].astype("Float64")
                b["_end_abs"]   = b["_start_abs"] + dn.astype(int)
                prev_end = None
                for i in order:
                    s = int(b.loc[i, "_start_abs"])
                    e = int(b.loc[i, "_end_abs"])
                    if prev_end is not None and s < prev_end:
                        shift = prev_end - s
                        s += shift
                        e += shift
                    b.loc[i, "_start_abs"] = s
                    b.loc[i, "_end_abs"] = e
                    prev_end = e
            else:
                if len(order) == 1:
                    b["_start_abs"] = b["_start_abs0"].astype("Float64")
                    b["_end_abs"]   = b["_end_abs0"].astype("Float64")
                else:
                    if total_gap0 > 0:
                        scaled = [g * slack / total_gap0 for g in gaps0]
                    else:
                        scaled = [slack / (len(order) - 1)] * (len(order) - 1)

                    gaps_int = [int(np.floor(x)) for x in scaled]
                    rem = int(slack - sum(gaps_int))
                    for k in range(rem):
                        gaps_int[k % (len(order) - 1)] += 1

                    cur = day_start
                    b["_start_abs"] = pd.NA
                    b["_end_abs"] = pd.NA

                    for pos, i in enumerate(order):
                        d = int(dn.loc[i])
                        s = cur
                        e = s + d
                        b.loc[i, "_start_abs"] = s
                        b.loc[i, "_end_abs"] = e
                        if pos < len(order) - 1:
                            cur = e + gaps_int[pos]

                    last_i = order[-1]
                    drift = int(b.loc[last_i, "_end_abs"]) - day_end
                    if drift != 0:
                        b.loc[last_i, "_end_abs"] = int(b.loc[last_i, "_end_abs"]) - drift
        else:
            b["_start_abs"] = b["_start_abs0"].astype("Float64")
            b["_end_abs"]   = b["_start_abs"] + dn.astype(int)
            prev_end = None
            for i in order:
                s = int(b.loc[i, "_start_abs"])
                e = int(b.loc[i, "_end_abs"])
                if prev_end is not None and s < prev_end:
                    shift = prev_end - s
                    s += shift
                    e += shift
                b.loc[i, "_start_abs"] = s
                b.loc[i, "_end_abs"] = e
                prev_end = e

        for i in valid_idx:
            if pd.isna(b.loc[i, "_start_abs"]) or pd.isna(b.loc[i, "_end_abs"]):
                continue
            b.loc[i, TIME_COL_START] = minutes_to_hhmmss_mod(b.loc[i, "_start_abs"])
            b.loc[i, TIME_COL_END]   = minutes_to_hhmmss_mod(b.loc[i, "_end_abs"])

        if duree_col is not None:
            b[duree_col] = pd.to_numeric(dur_new, errors="coerce").astype("Int64")

        for c in ["_start_abs", "_end_abs"]:
            if c in b.columns:
                b.drop(columns=c, inplace=True)

        return b.set_index(idx_colname).sort_index()

    processed_blocks = []
    for _, sub in df_work.groupby(["_jour", collab_col], dropna=False, sort=False):
        processed_blocks.append(process_block(sub))

    df_full = pd.concat(processed_blocks).sort_index()

    # -------------------- HARD DELTA GUARDS (THE KEY) --------------------
    # Recompute deltas globally, and enforce allowed-only receiving / 61010-only giving
    if duree_col is None or duree_col not in df_full.columns:
        raise ValueError("Need a Durée column for strict delta guards. (Find/rename Durée column.)")

    dur_in  = pd.to_numeric(df_work["_dur0"], errors="coerce").fillna(0).astype(int)
    dur_out = pd.to_numeric(df_full[duree_col], errors="coerce").fillna(0).astype(int)
    delta = dur_out - dur_in

    codes = df_full[code_col].apply(norm_code)

    bad_recv = df_full[(delta > 0) & (~codes.isin(ALLOWED_TARGET_CODES_NORM))].copy()
    bad_give = df_full[(delta < 0) & (codes != CODE_61010_NORM)].copy()

    print("\n=== STRICT GUARDS ===")
    print("Sum delta (must be 0):", int(delta.sum()))
    print("Receiving on non-allowed codes:", len(bad_recv))
    print("Giving on non-61010 codes:", len(bad_give))

    if not bad_recv.empty:
        print("\n⚠️ NOT ALLOWED: rows that RECEIVED minutes but code not in whitelist (showing 30):")
        display(bad_recv[[date_col, collab_col, code_col, TIME_COL_START, TIME_COL_END, duree_col]].head(30))
        raise ValueError("STOP: some minutes were added to codes outside the allowed list.")

    if not bad_give.empty:
        print("\n⚠️ NOT ALLOWED: rows that LOST minutes but code != 61010 (showing 30):")
        display(bad_give[[date_col, collab_col, code_col, TIME_COL_START, TIME_COL_END, duree_col]].head(30))
        raise ValueError("STOP: some minutes were removed from codes other than 61010.")

    # Quick recap by code
    recap = (
        df_full.assign(_code=df_full[code_col].apply(norm_code))
              .groupby("_code")[["Temps ajouté","Temps retiré"]].sum()
              .query("`Temps ajouté` > 0 or `Temps retiré` > 0")
              .sort_values(["Temps ajouté","Temps retiré"], ascending=False)
              .reset_index()
    )
    print("\n✅ Allocation recap (only codes involved should appear):")
    display(recap)

    # cleanup
    for c in ["_start_min", "_end_min", "_start_abs0", "_end_abs0", "_dur0", "_jour"]:
        if c in df_full.columns:
            df_full.drop(columns=c, inplace=True)

    final_cols = original_cols.copy()
    for extra in ["Temps retiré", "Temps ajouté"]:
        if extra not in final_cols:
            final_cols.append(extra)

    df_out = df_full.reindex(columns=final_cols)

# -------------------- FINAL HARD CHECK: Durée must equal Fin - Début --------------------
check = df_out.copy()

check["_start_check"] = check[TIME_COL_START].apply(to_min)
check["_end_check"]   = check[TIME_COL_END].apply(to_min)

check["_duration_from_times"] = check.apply(
    lambda r: diff_with_midnight_wrap(r["_end_check"], r["_start_check"]),
    axis=1
)

check["_duration_col"] = pd.to_numeric(check[duree_col], errors="coerce")

bad_duration = check[
    check["_duration_col"].fillna(-999999).astype(int)
    != check["_duration_from_times"].fillna(-999999).astype(int)
].copy()

print("\n=== FINAL DURATION CHECK ===")
print("Rows where Durée != Fin - Début:", len(bad_duration))

if not bad_duration.empty:
    display(
        bad_duration[
            [date_col, collab_col, code_col, TIME_COL_START, TIME_COL_END,
             duree_col, "_duration_from_times", "Temps retiré", "Temps ajouté"]
        ].head(50)
    )
    raise ValueError("STOP: Some exported rows have Durée different from Fin - Début.")

print("✅ Final check passed: every exported Durée equals Fin - Début.")
# Save
output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)
if output_path.suffix.lower() == ".xlsx":
    df_out.to_excel(output_path, index=False)
else:
    df_out.to_csv(output_path, index=False)

print(f"\n✅ Saved adjusted RAW RDA to:\n   {output_path}")
display(df_out.head(20))


Using prestation column : No prestation
Using date column       : Jour
Using collab column     : No collaborateur
Using durée column      : Durée

=== WHITELIST FORK ===
Whitelisted prestation codes diverted to 101: []
Rows diverted to 101 transfer path: 0
Rows kept in main 15-minute path: 3120

=== STRICT GUARDS ===
Sum delta (must be 0): 0
Receiving on non-allowed codes: 0
Giving on non-61010 codes: 0

✅ Allocation recap (only codes involved should appear):


,_code,Temps ajouté,Temps retiré
0,11200,2499,0
1,11100,1842,0
2,14200,9,0
3,61010,0,4350



=== FINAL DURATION CHECK ===
Rows where Durée != Fin - Début: 0
✅ Final check passed: every exported Durée equals Fin - Début.

✅ Saved adjusted RAW RDA to:
   /content/drive/MyDrive/NE_15minsreduit_April2026/ne_april (3)_61010_adjusted_no_overlap.xlsx


,Statut,Facturation,Jour,No collaborateur,Collaborateur,Durée,Début,Fin,N° du client,Client,No prestation,Prestation,UO,Date de création,Manière d'enregistrement,Durée_original,Temps retiré,Temps ajouté
index,,,,,,,,,,,,,,,,,,
0,OK,Non validé,01.04.2026,1209,Sihyürek Mine,15,10:28:00,10:43:00,NaN,,61010,Temps de déplacement soins,Home Assistance (Neuchatel) SA,01.04.2026 10:43:15,asebis smart,35,20,0
1,OK,Non validé,01.04.2026,1209,Sihyürek Mine,22,18:25:00,18:47:00,202863.0,LEUBA Erica,11100,OPAS-B Examens et traitements,Home Assistance (Neuchatel) SA,01.04.2026 18:52:46,asebis smart,22,0,0
2,OK,Non validé,01.04.2026,1209,Sihyürek Mine,15,09:17:00,09:32:00,NaN,,61010,Temps de déplacement soins,Home Assistance (Neuchatel) SA,01.04.2026 09:30:39,asebis smart,26,11,0
3,OK,Non validé,13.04.2026,1209,Sihyürek Mine,20,11:27:00,11:47:00,104115.0,BOSCOE EDITH,11100,OPAS-B Examens et traitements,Home Assistance (Neuchatel) SA,13.04.2026 12:48:35,asebis smart,20,0,0
4,OK,Non validé,21.04.2026,1209,Sihyürek Mine,15,18:46:00,19:01:00,NaN,,61010,Temps de déplacement soins,Home Assistance (Neuchatel) SA,21.04.2026 19:01:47,asebis smart,21,6,0
5,OK,Non validé,01.04.2026,1209,Sihyürek Mine,30,10:43:00,11:13:00,202896.0,JEAN-MAIRET Patrice,11100,OPAS-B Examens et traitements,Home Assistance (Neuchatel) SA,01.04.2026 11:03:34,asebis smart,15,0,15
6,OK,Non validé,01.04.2026,1209,Sihyürek Mine,15,15:55:00,16:10:00,NaN,,61010,Temps de déplacement soins,Home Assistance (Neuchatel) SA,01.04.2026 16:28:36,asebis smart,28,13,0
7,OK,Non validé,01.04.2026,1209,Sihyürek Mine,15,08:31:00,08:46:00,NaN,,61010,Temps de déplacement soins,Home Assistance (Neuchatel) SA,01.04.2026 08:52:47,asebis smart,20,5,0
8,OK,Non validé,01.04.2026,1209,Sihyürek Mine,40,07:19:00,07:59:00,202940.0,MILAZZO Filippo,11100,OPAS-B Examens et traitements,Home Assistance (Neuchatel) SA,01.04.2026 07:46:42,asebis smart,25,0,15


In [ ]:
#@title ✅ Export Excel: Differences in start/end times (RAW df vs ADJUSTED df_out)

import pandas as pd
import numpy as np
from pathlib import Path
import datetime as dt

# ----------------------------
# 0) Get RAW + ADJUSTED dfs
# ----------------------------
def _load_df_from_path(p: Path) -> pd.DataFrame:
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")
    if p.suffix.lower() == ".xlsx":
        return pd.read_excel(p)
    elif p.suffix.lower() == ".csv":
        return pd.read_csv(p)
    else:
        raise ValueError(f"Unsupported file type: {p.suffix} (use .xlsx or .csv)")

# RAW (main path only; whitelisted rows are intentionally excluded from df_out)
if "df_main_raw" in globals():
    raw_df = df_main_raw.copy()
elif "df" in globals():
    raw_df = df.copy()
else:
    if "input_path" in globals():
        raw_df = _load_df_from_path(Path(input_path))
    elif "EXCEL_OR_CSV_FILE" in globals():
        raw_df = _load_df_from_path(Path(EXCEL_OR_CSV_FILE))
    else:
        raise NameError("Cannot find RAW df. Run Cell 1 or define input_path / EXCEL_OR_CSV_FILE.")

# ADJUSTED
if "df_out" in globals():
    adj_df = df_out.copy()
else:
    if "OUTPUT_FILE" in globals():
        adj_df = _load_df_from_path(Path(OUTPUT_FILE))
    else:
        raise NameError("Cannot find adjusted df_out / OUTPUT_FILE. Run Cell 2 first.")

print(f"RAW rows: {len(raw_df):,} | ADJUSTED rows: {len(adj_df):,}")

# ----------------------------
# 1) Column detection
# ----------------------------
DATE_COL_CANDIDATES   = ['Date Début', 'Date', 'Jour', 'Date de prestation']
COLLAB_COL_CANDIDATES = ['No collaborateur', 'Collaborateur', 'ID collaborateur']
START_COL_CANDIDATES  = ['Début', 'Heure début', 'Heure Début']
END_COL_CANDIDATES    = ['Fin', 'Heure fin', 'Heure Fin']
CODE_COL_CANDIDATES   = ['N° Prestation','No prestation','Code prestation','Prestation','Code','N° Prestation']

def pick_col(df_, candidates):
    return next((c for c in candidates if c in df_.columns), None)

raw_date_col   = pick_col(raw_df, DATE_COL_CANDIDATES)
raw_collab_col = pick_col(raw_df, COLLAB_COL_CANDIDATES)
raw_start_col  = pick_col(raw_df, START_COL_CANDIDATES)
raw_end_col    = pick_col(raw_df, END_COL_CANDIDATES)
raw_code_col   = pick_col(raw_df, CODE_COL_CANDIDATES)

adj_date_col   = pick_col(adj_df, DATE_COL_CANDIDATES)
adj_collab_col = pick_col(adj_df, COLLAB_COL_CANDIDATES)
adj_start_col  = pick_col(adj_df, START_COL_CANDIDATES)
adj_end_col    = pick_col(adj_df, END_COL_CANDIDATES)
adj_code_col   = pick_col(adj_df, CODE_COL_CANDIDATES)

required = [
    ("RAW date", raw_date_col), ("RAW collab", raw_collab_col), ("RAW start", raw_start_col), ("RAW end", raw_end_col),
    ("ADJ date", adj_date_col), ("ADJ collab", adj_collab_col), ("ADJ start", adj_start_col), ("ADJ end", adj_end_col),
]
missing = [name for name, col in required if col is None]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Using columns:")
print(f"  RAW: date={raw_date_col}, collab={raw_collab_col}, start={raw_start_col}, end={raw_end_col}, code={raw_code_col}")
print(f"  ADJ: date={adj_date_col}, collab={adj_collab_col}, start={adj_start_col}, end={adj_end_col}, code={adj_code_col}")

# ----------------------------
# 2) Time parsing helpers
# ----------------------------
def to_min_any(t):
    """Minutes since 00:00. Supports Timestamp/time, Excel float, 'HH:MM[:SS]' and allows hours > 23 (e.g. 25:10)."""
    if pd.isna(t):
        return np.nan

    if isinstance(t, pd.Timestamp):
        return int(t.hour) * 60 + int(t.minute)
    if isinstance(t, dt.time):
        return int(t.hour) * 60 + int(t.minute)

    if isinstance(t, (float, int)) and not isinstance(t, bool):
        x = float(t)
        # Excel time fraction of day
        if 0 <= x < 1.0:
            return int(round(x * 24 * 60))
        # minutes since midnight
        if 0 <= x < 24 * 60 and abs(x - round(x)) < 1e-9:
            return int(round(x))
        # hours decimal
        if 0 <= x < 1000:
            return int(round(x * 60))

    s = str(t).strip()
    # try datetime parse
    dtv = pd.to_datetime(s, dayfirst=True, errors='coerce')
    if not pd.isna(dtv):
        return int(dtv.hour) * 60 + int(dtv.minute)

    # fallback "HH:MM[:SS]" (HH can be > 23)
    tok = s.split()[-1]
    parts = tok.split(':')
    if len(parts) >= 2 and parts[0].lstrip("+-").isdigit() and parts[1].isdigit():
        return int(parts[0]) * 60 + int(parts[1])

    return np.nan

def end_abs(end_min, start_min):
    """If an interval wraps midnight (end < start), treat end as end+1440 for comparisons."""
    if pd.isna(end_min) or pd.isna(start_min):
        return np.nan
    e = float(end_min)
    s = float(start_min)
    return e if e >= s else e + 1440.0

def min_to_hhmm(m):
    if pd.isna(m):
        return ""
    m = int(round(float(m)))
    h = m // 60
    mm = m % 60
    return f"{h:02d}:{mm:02d}"

# ----------------------------
# 3) Build normalized tables with helper cols
# ----------------------------
def prep(df_, date_col, collab_col, start_col, end_col, code_col, label):
    out = df_.copy()
    out["_row_id"] = np.arange(len(out))  # stable line id based on row order in file/dataframe
    out["_date"] = pd.to_datetime(out[date_col], dayfirst=True, errors="coerce").dt.date
    out["_collab"] = out[collab_col]
    out["_start_min"] = out[start_col].apply(to_min_any)
    out["_end_min"] = out[end_col].apply(to_min_any)
    out["_end_abs"] = out.apply(lambda r: end_abs(r["_end_min"], r["_start_min"]), axis=1)
    if code_col and code_col in out.columns:
        out["_code"] = out[code_col].astype(str).str.strip()
    else:
        out["_code"] = ""
    out["_label"] = label
    return out

raw = prep(raw_df, raw_date_col, raw_collab_col, raw_start_col, raw_end_col, raw_code_col, "RAW")
adj = prep(adj_df, adj_date_col, adj_collab_col, adj_start_col, adj_end_col, adj_code_col, "ADJUSTED")

# ----------------------------
# 4) DAILY summary per (date, collab)
# ----------------------------
raw_daily = (
    raw.dropna(subset=["_date","_collab"])
       .groupby(["_date","_collab"], dropna=False)
       .agg(raw_day_start_min=("_start_min","min"),
            raw_day_end_abs=("_end_abs","max"),
            raw_rows=(" _row_id".strip(), "count") if " _row_id".strip() in raw.columns else ("_row_id","count"))
       .reset_index()
)
adj_daily = (
    adj.dropna(subset=["_date","_collab"])
       .groupby(["_date","_collab"], dropna=False)
       .agg(adj_day_start_min=("_start_min","min"),
            adj_day_end_abs=("_end_abs","max"),
            adj_rows=("_row_id","count"))
       .reset_index()
)

daily = pd.merge(raw_daily, adj_daily, on=["_date","_collab"], how="outer")

daily["raw_day_start"] = daily["raw_day_start_min"].apply(min_to_hhmm)
daily["adj_day_start"] = daily["adj_day_start_min"].apply(min_to_hhmm)
daily["raw_day_end"]   = daily["raw_day_end_abs"].apply(min_to_hhmm)
daily["adj_day_end"]   = daily["adj_day_end_abs"].apply(min_to_hhmm)

daily["delta_start_min"] = daily["adj_day_start_min"] - daily["raw_day_start_min"]
daily["delta_end_min"]   = daily["adj_day_end_abs"]   - daily["raw_day_end_abs"]

daily_changed = daily[
    (daily["delta_start_min"].fillna(0) != 0) |
    (daily["delta_end_min"].fillna(0) != 0)
].copy()

daily_changed = daily_changed.sort_values(["_date","_collab"])

# ----------------------------
# 5) ROW-level diffs (RDAs where Début/Fin changed)
# ----------------------------
rows_unmatched = pd.DataFrame()

if len(raw) == len(adj):
    # direct row_id alignment (best case)
    merged = pd.merge(
        raw[["_row_id","_date","_collab","_code","_start_min","_end_min","_end_abs"]],
        adj[["_row_id","_start_min","_end_min","_end_abs"]],
        on="_row_id",
        how="inner",
        suffixes=('_raw','_adj') # suffixes apply to common cols other than 'on' key
    )

else:
    # fallback: match within each (date, collab) by sorted sequence
    raw2 = raw.copy()
    adj2 = adj.copy()

    raw2["_seq"] = raw2.sort_values(["_date","_collab","_start_min","_end_abs","_code"], na_position="last") \
                      .groupby(["_date","_collab"], dropna=False).cumcount()
    adj2["_seq"] = adj2.sort_values(["_date","_collab","_start_min","_end_abs","_code"], na_position="last") \
                      .groupby(["_date","_collab"], dropna=False).cumcount()

    merged = pd.merge(
        raw2[["_row_id","_date","_collab","_code","_seq","_start_min","_end_min","_end_abs"]],
        adj2[["_row_id","_date","_collab","_code","_seq","_start_min","_end_min","_end_abs"]],
        on=["_date","_collab","_seq"],
        how="outer",
        suffixes=('_raw','_adj'),
        indicator=True
    )
    rows_unmatched = merged[merged["_merge"] != "both"].copy()
    merged = merged[merged["_merge"] == "both"].copy()

merged["raw_start"] = merged["_start_min_raw"].apply(min_to_hhmm)
merged["adj_start"] = merged["_start_min_adj"].apply(min_to_hhmm)
merged["raw_end"]   = merged["_end_min_raw"].apply(min_to_hhmm)
merged["adj_end"]   = merged["_end_min_adj"].apply(min_to_hhmm)

merged["delta_start_min"] = merged["_start_min_adj"] - merged["_start_min_raw"]
merged["delta_end_min"]   = merged["_end_abs_adj"]   - merged["_end_abs_raw"]

rows_changed = merged[
    (merged["delta_start_min"].fillna(0) != 0) |
    (merged["delta_end_min"].fillna(0) != 0)
].copy()

# Dynamically build the list of columns to keep
cols_to_select = ["_date", "_collab"]

# Handle _code column(s)
if "_code_raw" in rows_changed.columns and "_code_adj" in rows_changed.columns:
    cols_to_select.extend(["_code_raw", "_code_adj"])
elif "_code" in rows_changed.columns:
    cols_to_select.append("_code")

cols_to_select.extend([
    "raw_start", "raw_end", "adj_start", "adj_end",
    "delta_start_min", "delta_end_min"
])

# Handle _row_id column(s)
if "_row_id_raw" in rows_changed.columns and "_row_id_adj" in rows_changed.columns:
    cols_to_select.extend(["_row_id_raw", "_row_id_adj"])
elif "_row_id" in rows_changed.columns:
    cols_to_select.append("_row_id")

# Handle _seq column (only present in the else branch)
if "_seq" in rows_changed.columns:
    cols_to_select.append("_seq")

rows_changed = rows_changed[cols_to_select].sort_values(["_date","_collab"])

# ----------------------------
# 6) Write Excel
# ----------------------------
# where to save
if "root_folder" in globals():
    out_path_drive = Path(root_folder) / "RDA_StartEnd_Differences_RAW_vs_ADJ.xlsx"
else:
    out_path_drive = Path("RDA_StartEnd_Differences_RAW_vs_ADJ.xlsx")

# also save a copy locally for download (optional)
out_path_local = Path("/content") / out_path_drive.name if Path("/content").exists() else Path(out_path_drive.name)

meta = pd.DataFrame([
    ["RAW rows", len(raw_df)],
    ["ADJUSTED rows", len(adj_df)],
    ["Daily changed rows", len(daily_changed)],
    ["Row-level changed RDAs", len(rows_changed)],
    ["Unmatched rows (if any)", len(rows_unmatched)],
    ["RAW date col", raw_date_col],
    ["RAW collab col", raw_collab_col],
    ["RAW start/end col", f"{raw_start_col} / {raw_end_col}"],
    ["ADJ date col", adj_date_col],
    ["ADJ collab col", adj_collab_col],
    ["ADJ start/end col", f"{adj_start_col} / {adj_end_col}"],
    ["Saved (Drive/local path)", str(out_path_drive)],
], columns=["Metric","Value"])

with pd.ExcelWriter(out_path_drive, engine="openpyxl") as writer:
    meta.to_excel(writer, sheet_name="Meta", index=False)
    daily_changed.to_excel(writer, sheet_name="Daily_StartEnd_Changed", index=False)
    rows_changed.to_excel(writer, sheet_name="RDAs_StartEnd_Changed", index=False)
    if not rows_unmatched.empty:
        rows_unmatched.to_excel(writer, sheet_name="Unmatched_Rows", index=False)

print(f"\n✅ Excel created:\n   {out_path_drive}")

# If /content exists, also copy for easy download in Colab file browser
try:
    if out_path_local != out_path_drive:
        import shutil
        shutil.copyfile(out_path_drive, out_path_local)
        print(f"✅ Copy for download:\n   {out_path_local}")
except Exception as e:
    print("Note: Could not copy to /content:", e)

# Quick peek
display(meta)
display(daily_changed.head(20))
display(rows_changed.head(20))
if not rows_unmatched.empty:
    print("\n⚠️ There were unmatched rows (RAW vs ADJ) — see sheet 'Unmatched_Rows'.")


RAW rows: 3,120 | ADJUSTED rows: 3,120
Using columns:
  RAW: date=Jour, collab=No collaborateur, start=Début, end=Fin, code=No prestation
  ADJ: date=Jour, collab=No collaborateur, start=Début, end=Fin, code=No prestation

✅ Excel created:
   /content/drive/MyDrive/NE_15minsreduit_April2026/RDA_StartEnd_Differences_RAW_vs_ADJ.xlsx
✅ Copy for download:
   /content/RDA_StartEnd_Differences_RAW_vs_ADJ.xlsx


,Metric,Value
0,RAW rows,3120
1,ADJUSTED rows,3120
2,Daily changed rows,0
3,Row-level changed RDAs,1154
4,Unmatched rows (if any),0
5,RAW date col,Jour
6,RAW collab col,No collaborateur
7,RAW start/end col,Début / Fin
8,ADJ date col,Jour
9,ADJ collab col,No collaborateur


,_date,_collab,raw_day_start_min,raw_day_end_abs,raw_rows,adj_day_start_min,adj_day_end_abs,adj_rows,raw_day_start,adj_day_start,raw_day_end,adj_day_end,delta_start_min,delta_end_min


,_date,_collab,_code,raw_start,raw_end,adj_start,adj_end,delta_start_min,delta_end_min,_row_id
0,2026-04-01,1209,61010,10:08,10:43,10:28,10:43,20,0.0,0
2,2026-04-01,1209,61010,09:06,09:32,09:17,09:32,11,0.0,2
5,2026-04-01,1209,11100,10:43,10:58,10:43,11:13,0,15.0,5
6,2026-04-01,1209,61010,15:55,16:23,15:55,16:10,0,-13.0,6
7,2026-04-01,1209,61010,08:26,08:46,08:31,08:46,5,0.0,7
8,2026-04-01,1209,11100,07:19,07:44,07:19,07:59,0,15.0,8
9,2026-04-01,1209,11200,08:14,08:26,08:14,08:31,0,5.0,9
11,2026-04-01,1209,11100,16:49,17:10,16:49,17:22,0,12.0,11
12,2026-04-01,1209,11100,16:23,16:44,16:10,16:44,-13,0.0,12
13,2026-04-01,1209,61010,17:10,17:37,17:22,17:37,12,0.0,13


In [ ]:
# @title
# === Cell 3 — Check for overlaps per collaborateur & day (own schedule only) ===

from pathlib import Path
from IPython.display import display

# ----------------- Get the adjusted DataFrame -----------------
try:
    df_check = df_out.copy()
    print("Using in-memory df_out from Cell 2.")
except NameError:
    try:
        OUTPUT_FILE
    except NameError:
        raise NameError("Run Cells 1 and 2 first so 'df_out' or 'OUTPUT_FILE' exist.")
    ext = Path(OUTPUT_FILE).suffix.lower()
    if ext == ".xlsx":
        df_check = pd.read_excel(OUTPUT_FILE)
    else:
        df_check = pd.read_csv(OUTPUT_FILE)
    print(f"Reloaded adjusted file from: {OUTPUT_FILE}")

# ----------------- Basic config / column detection -----------------
TIME_COL_START = 'Début'
TIME_COL_END   = 'Fin'

DATE_COL_CANDIDATES   = ['Date Début', 'Date', 'Jour', 'Date de prestation']
COLLAB_COL_CANDIDATES = ['No collaborateur', 'Collaborateur', 'ID collaborateur']
CODE_COL_CANDIDATES   = ['No prestation', 'Code prestation', 'Prestation', 'Code']

for col in (TIME_COL_START, TIME_COL_END):
    if col not in df_check.columns:
        raise KeyError(f"Missing required time column in adjusted file: {col!r}")

date_col_overlap = next((c for c in DATE_COL_CANDIDATES if c in df_check.columns), None)
if date_col_overlap is None:
    raise ValueError(f"Could not find a date column; tried: {DATE_COL_CANDIDATES}")

collab_col_overlap = next((c for c in COLLAB_COL_CANDIDATES if c in df_check.columns), None)
if collab_col_overlap is None:
    raise ValueError(f"Could not find a collaborator column; tried: {COLLAB_COL_CANDIDATES}")

code_col_overlap = next((c for c in CODE_COL_CANDIDATES if c in df_check.columns), None)

print(f"Overlap check using date column:   {date_col_overlap}")
print(f"Overlap check using collab column: {collab_col_overlap}")
print(f"Overlap check using code column:   {code_col_overlap}")

# ----------------- Time helper (minutes since midnight) -----------------
import datetime as dt

def to_min_for_check(t):
    if pd.isna(t):
        return np.nan
    if isinstance(t, pd.Timestamp):
        return int(t.hour) * 60 + int(t.minute)
    if isinstance(t, dt.time):
        return int(t.hour) * 60 + int(t.minute)
    if isinstance(t, (float, int)) and not isinstance(t, bool):
        x = float(t)
        if 0 <= x < 1.0:
            return int(round(x * 24 * 60)) % (24 * 60)
        if 0 <= x < 24 * 60 and abs(x - round(x)) < 1e-9:
            return int(round(x))
        if 0 <= x < 24:
            return int(round(x * 60))

    s = str(t).strip()
    dtv = pd.to_datetime(s, dayfirst=True, errors='coerce')
    if not pd.isna(dtv):
        return int(dtv.hour) * 60 + int(dtv.minute)

    tok = s.split()[-1]
    parts = tok.split(':')
    if len(parts) < 2:
        raise ValueError(f"Unexpected time format in adjusted file: {t!r}")
    return int(parts[0]) * 60 + int(parts[1])


# Build helper columns
df_check['_start_min_chk'] = df_check[TIME_COL_START].apply(to_min_for_check)
df_check['_end_min_chk']   = df_check[TIME_COL_END].apply(to_min_for_check)

dts_chk = pd.to_datetime(df_check[date_col_overlap], dayfirst=True, errors='coerce')
df_check['_jour_chk'] = dts_chk.dt.date

# ----------------- Scan for overlaps per (jour, collab) -----------------
overlaps = []

for (jour, collab), sub in df_check.groupby(['_jour_chk', collab_col_overlap], dropna=False, sort=False):
    b = sub.sort_values(by=['_start_min_chk', '_end_min_chk']).copy()
    b = b.reset_index()  # keep original index in 'index'

    for i in range(len(b) - 1):
        s1 = b.loc[i, '_start_min_chk']
        e1 = b.loc[i, '_end_min_chk']
        s2 = b.loc[i+1, '_start_min_chk']
        e2 = b.loc[i+1, '_end_min_chk']

        if pd.isna(s1) or pd.isna(e1) or pd.isna(s2) or pd.isna(e2):
            continue

        # Strict overlap if first ends AFTER second starts
        if e1 > s2:
            overlaps.append({
                'Date'            : jour,
                'Collaborateur'   : collab,
                'Row1_index_orig' : b.loc[i, 'index'],
                'Row1_Début'      : b.loc[i, TIME_COL_START],
                'Row1_Fin'        : b.loc[i, TIME_COL_END],
                'Row1_Code'       : b.loc[i][code_col_overlap] if code_col_overlap else None,
                'Row2_index_orig' : b.loc[i+1, 'index'],
                'Row2_Début'      : b.loc[i+1, TIME_COL_START],
                'Row2_Fin'        : b.loc[i+1, TIME_COL_END],
                'Row2_Code'       : b.loc[i+1][code_col_overlap] if code_col_overlap else None,
                'Overlap_minutes' : e1 - s2
            })

# Keep overlaps_df for dashboard
if overlaps:
    overlaps_df = pd.DataFrame(overlaps)
else:
    overlaps_df = pd.DataFrame(columns=[
        'Date', 'Collaborateur', 'Row1_index_orig', 'Row1_Début', 'Row1_Fin',
        'Row1_Code', 'Row2_index_orig', 'Row2_Début', 'Row2_Fin',
        'Row2_Code', 'Overlap_minutes'
    ])

# ----------------- Report results -----------------
if overlaps_df.empty:
    print("\n✅ No overlaps detected within any single collaborateur's schedule for a given day.")
else:
    print(f"\n⚠️ Found {len(overlaps_df)} overlapping interval pairs "
          f"(same collab + same day, where Row1.Fin > Row2.Début). Showing first 20:")
    display(overlaps_df.head(20))

# Cleanup helper cols
for c in ['_start_min_chk', '_end_min_chk', '_jour_chk']:
    if c in df_check.columns:
        df_check.drop(columns=c, inplace=True)


Using in-memory df_out from Cell 2.
Overlap check using date column:   Jour
Overlap check using collab column: No collaborateur
Overlap check using code column:   No prestation

✅ No overlaps detected within any single collaborateur's schedule for a given day.


In [ ]:
# @title
# === Cell 4 — Generate structured Nexus exports:
#   1) One CSV + batch for ALL collabs
#   2) One CSV + batch for collabs that have any 61010
#   3) Per-collab folders, each with its own CSV + batch

from pathlib import Path
from IPython.display import display

# Make sure Drive is mounted (safe if already)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ----------------- Get the adjusted DataFrame -----------------
try:
    df_batch = df_out.copy() # Use adjusted_df here for consistency with other parts of the flow
    print("Using in-memory adjusted_df from Cell 2.")
except NameError:
    try:
        OUTPUT_FILE
    except NameError:
        raise NameError("Run Cells 1 and 2 first so 'adjusted_df' or 'OUTPUT_FILE' exist.")
    out_path = Path(OUTPUT_FILE)
    if out_path.suffix.lower() == ".xlsx":
        df_batch = pd.read_excel(out_path)
    else:
        df_batch = pd.read_csv(out_path)
    print(f"Reloaded adjusted file from: {OUTPUT_FILE}")

print("\nColumns in adjusted RDA (for batch export):")
print(df_batch.columns.tolist())

# ----------------- Column detection -----------------
DATE_COL_CANDIDATES    = ['Date Début', 'Date', 'Jour', 'Date de prestation']
START_COL_CANDIDATES   = ['Début', 'Heure début', 'Heure Début']
END_COL_CANDIDATES     = ['Fin', 'Heure fin', 'Heure Fin']
PREST_COL_CANDIDATES   = ['N° Prestation', 'No prestation', 'Code prestation', 'Prestation', 'Code'] # Added 'N° Prestation'
DUREE_COL_CANDIDATES   = ['Durée', 'Duree', 'Durée (min)', 'Durée (minutes)']
CLIENT_COL_CANDIDATES  = ['N° du client', 'No client', 'ID client', 'Client']
COLLAB_COL_CANDIDATES  = ['No collaborateur', 'Collaborateur', 'ID collaborateur']

date_col   = next((c for c in DATE_COL_CANDIDATES   if c in df_batch.columns), None)
start_col  = next((c for c in START_COL_CANDIDATES  if c in df_batch.columns), None)
end_col    = next((c for c in END_COL_CANDIDATES    if c in df_batch.columns), None)
prest_col  = next((c for c in PREST_COL_CANDIDATES  if c in df_batch.columns), None)
duree_col  = next((c for c in DUREE_COL_CANDIDATES  if c in df_batch.columns), None)
client_col = next((c for c in CLIENT_COL_CANDIDATES if c in df_batch.columns), None)
collab_col = next((c for c in COLLAB_COL_CANDIDATES if c in df_batch.columns), None)

collab_name_col = next((c for c in ['Collaborateur', 'Nom Collaborateur']
                        if c in df_batch.columns and c != collab_col), None)

missing = [name for name, col in [
    ("date", date_col),
    ("start", start_col),
    ("end", end_col),
    ("prestation", prest_col),
    ("durée", duree_col),
    ("client", client_col),
    ("collaborateur", collab_col),
] if col is None]

if missing:
    raise ValueError(f"Missing required columns for batch export: {missing}")

print("\nUsing columns:")
print(f"  Date          : {date_col}")
print(f"  Début         : {start_col}")
print(f"  Fin           : {end_col}")
print(f"  Prestation    : {prest_col}")
print(f"  Durée         : {duree_col}")
print(f"  N° du client  : {client_col}")
print(f"  Collaborateur : {collab_col}")
if collab_name_col:
    print(f"  Collaborateur Name : {collab_name_col}")

# ----------------- Helpers -----------------
def format_time(t):
    """
    Format Début/Fin for Nexus CSV: return 'HH:MM'.
    Handles datetime, 'HH:MM:SS', Excel floats, etc.
    """
    if pd.isna(t):
        return ''
    if isinstance(t, pd.Timestamp):
        return t.strftime('%H:%M')
    try:
        dt = pd.to_datetime(t, errors='raise', dayfirst=True)
        return dt.strftime('%H:%M')
    except Exception:
        pass
    if isinstance(t, (float, int)):
        hours, remainder = divmod(float(t) * 24, 1)
        minutes = int(round(remainder * 60))
        return f"{int(hours):02d}:{minutes:02d}"
    s = str(t).strip()
    tok = s.split()[-1]
    parts = tok.split(':')
    if len(parts) >= 2:
        return f"{int(parts[0]):02d}:{int(parts[1]):02d}"
    return s[:5]

def format_date(d):
    """Format date as 'dd.mm.yyyy' for Nexus CSV."""
    dt = pd.to_datetime(d, dayfirst=True, errors='coerce')
    if pd.isna(dt):
        return '' # Return empty string for NaT values
    return dt.strftime('%d.%m.%Y')

def safe_folder_name(name):
    return "".join(c if c.isalnum() or c in " ._ +-" else "_" for c in str(name))

def to_nexus_csv(df_src: pd.DataFrame) -> pd.DataFrame:
    """
    Convert a slice of df_batch into the Nexus CSV schema.
    Uses OE_VALUE selected in Cell 0.
    """
    data = {
        'Datum'           : df_src[date_col].apply(format_date),
        'Von'             : df_src[start_col].apply(format_time),
        'Bis'             : df_src[end_col].apply(format_time),
        'Leistungscode'   : df_src[prest_col],
        'Dauer_verrechnet': df_src[duree_col],
        'OE'              : OE_VALUE,
        'KD-Nr'           : df_src[client_col].fillna(0).astype(int),
        'Klient'          : 0,
        'Einsatzgrund'    : df_src[client_col].fillna(0).apply(lambda x: 0 if x == 0 else 2),
        'Mitarbeiter-ID'  : df_src[collab_col],
    }
    out = pd.DataFrame(data)
    out['KD-Nr']           = out['KD-Nr'].astype(int)
    out['Leistungscode']   = out['Leistungscode'].astype(str)
    out['Dauer_verrechnet']= out['Dauer_verrechnet'].fillna(0).astype(int)
    out['Klient']          = out['Klient'].astype(int)
    out['Einsatzgrund']    = out['Einsatzgrund'].astype(int)
    out['Mitarbeiter-ID']  = out['Mitarbeiter-ID'].astype(int)
    return out

def write_batch_file(batch_path: str, csv_name: str, oe: str = None, map_rel: str = "..\\HAS_map_main.csv"):
    """
    Write a Windows .bat file to import the given CSV via Nexus client.
    Standard 201 exports use HAS_map_main.csv so the 101 transfer can own HAS_map.csv.
    """
    if oe is None:
        oe = OE_VALUE
    batch_content = (
        "@echo off\n"
        "chcp 65001\n"
        f"\"..\\nx-spi-client\\Asebis.Client.StarterCommand.exe\" "
        f"/u=nexus /p=fAvNCDnW3E /t=ImportLeistungen_CSV /o={oe} "
        f"/f=\"{csv_name}\" /map=\"{map_rel}\" /v\n"
        "Pause\n"
    )
    with open(batch_path, 'w', encoding='utf-8') as f:
        f.write(batch_content)

# ----------------- Root folder for all exports -----------------
# Root folder is now defined in Cell 1
# os.makedirs(root_folder, exist_ok=True) # This is also done in Cell 1

print(f"\nExport root folder:\n  {root_folder}")
print(f"OE used for exports: {OE_VALUE}")

# ----------------- Durée summary per collab -----------------
if collab_name_col:
    duree_summary = df_batch.groupby([collab_col, collab_name_col])[duree_col].sum().reset_index()
    duree_summary.columns = ['Collaborateur_ID', 'Collaborateur_Name', 'Sum_Duree']
else:
    duree_summary = df_batch.groupby(collab_col)[duree_col].sum().reset_index()
    duree_summary.columns = ['Collaborateur_ID', 'Sum_Duree']

duree_summary_path = os.path.join(root_folder, 'RDA_duree_check.csv')
duree_summary.to_csv(duree_summary_path, index=False, sep=';') # Changed separator to semicolon
print(f"Saved durée summary to: {duree_summary_path}")

# ============================================================
# 1) FOLDER A — ONE BIG CSV FOR ALL COLLABS
# ============================================================
folder_all = os.path.join(root_folder, "01_All_Collabs_One_CSV")
os.makedirs(folder_all, exist_ok=True)

all_duree = int(df_batch[duree_col].fillna(0).sum())
all_csv_name = f"RDA_AllCollabs+{all_duree}.csv"
all_csv_path = os.path.join(folder_all, all_csv_name)

all_csv_df = to_nexus_csv(df_batch)
all_csv_df.to_csv(all_csv_path, index=False, sep=';')

all_batch_name = "RDA_AllCollabs_batch.bat"
all_batch_path = os.path.join(folder_all, all_batch_name)
write_batch_file(all_batch_path, all_csv_name)

print(f"\n[1] All-collabs export written to:\n  {all_csv_path}\n  {all_batch_path}")

# ============================================================
# 2) FOLDER B — ONE CSV FOR COLLABS THAT HAVE ANY 61010
# ============================================================
CODE_61010 = '61010'
mask_61010 = df_batch[prest_col].astype(str).str.strip() == CODE_61010
collabs_with_61010 = df_batch.loc[mask_61010, collab_col].unique().tolist()

folder_61010 = os.path.join(root_folder, "02_Collabs_With_61010_One_CSV")
os.makedirs(folder_61010, exist_ok=True)

if collabs_with_61010:
    df_61010_collabs = df_batch[df_batch[collab_col].isin(collabs_with_61010)].copy()
    duree_61010_collabs = int(df_61010_collabs[duree_col].fillna(0).sum())
    csv_61010_name = f"RDA_CollabsWith61010+{duree_61010_collabs}.csv"
    csv_61010_path = os.path.join(folder_61010, csv_61010_name)

    csv_61010_df = to_nexus_csv(df_61010_collabs)
    csv_61010_df.to_csv(csv_61010_path, index=False, sep=';')

    batch_61010_name = "RDA_CollabsWith61010_batch.bat"
    batch_61010_path = os.path.join(folder_61010, batch_61010_name)
    write_batch_file(batch_61010_path, csv_61010_name)

    print(f"\n[2] Collabs-with-61010 export written to:\n  {csv_61010_path}\n  {batch_61010_path}")
    print(f"    Collaborateurs included (have >=1 61010): {collabs_with_61010}")
else:
    print("\n[2] No collaborators with 61010 found — folder created but no CSV/batch generated.")

# ============================================================
# 3) FOLDER C — PER-COLLAB FOLDERS (ONE CSV + BATCH EACH)
# ============================================================
folder_per_collab = os.path.join(root_folder, "03_Per_Collab_Separate")
os.makedirs(folder_per_collab, exist_ok=True)

for collab_id, group in df_batch.groupby(collab_col):
    sum_duree = int(group[duree_col].fillna(0).sum())
    collab_safe_id = safe_folder_name(collab_id)

    collab_name = ""
    if collab_name_col and not group[collab_name_col].empty:
        collab_name = group[collab_name_col].iloc[0]
        collab_name_safe = safe_folder_name(collab_name)
        collab_identifier = f"{collab_safe_id}-{collab_name_safe}"
    else:
        collab_identifier = collab_safe_id

    collab_folder_name = f"RDA-{collab_identifier}+{sum_duree}"
    collab_folder = os.path.join(folder_per_collab, collab_folder_name)
    os.makedirs(collab_folder, exist_ok=True)

    csv_name = f"{collab_identifier}+{sum_duree}.csv"
    csv_path = os.path.join(collab_folder, csv_name)

    out_df = to_nexus_csv(group)
    out_df.to_csv(csv_path, index=False, sep=';')

    batch_name = f"{collab_identifier}_batch.bat"
    batch_path = os.path.join(collab_folder, batch_name)
    write_batch_file(batch_path, csv_name, map_rel="..\\..\\HAS_map_main.csv")

print(f"\n[3] Per-collaborateur CSVs + batch files written under:\n  {folder_per_collab}")

print("\n✅ Structured export complete.")
display(duree_summary.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using in-memory adjusted_df from Cell 2.

Columns in adjusted RDA (for batch export):
['Statut', 'Facturation', 'Jour', 'No collaborateur', 'Collaborateur', 'Durée', 'Début', 'Fin', 'N° du client', 'Client', 'No prestation', 'Prestation', 'UO', 'Date de création', "Manière d'enregistrement", 'Durée_original', 'Temps retiré', 'Temps ajouté']

Using columns:
  Date          : Jour
  Début         : Début
  Fin           : Fin
  Prestation    : No prestation
  Durée         : Durée
  N° du client  : N° du client
  Collaborateur : No collaborateur
  Collaborateur Name : Collaborateur

Export root folder:
  /content/drive/MyDrive/NE_15minsreduit_April2026
OE used for exports: 100000000000000301
Saved durée summary to: /content/drive/MyDrive/NE_15minsreduit_April2026/RDA_duree_check.csv

[1] All-collabs export written to:
  /content/drive/MyDrive/NE_15minsreduit_Ap

,Collaborateur_ID,Collaborateur_Name,Sum_Duree
0,1209,Sihyürek Mine,7234
1,2004,Incerti Mélanie,4792
2,2008,Le Pavec Anna,6979
3,2012,Belbahri Hakima,7732
4,2014,Mojsilovic Tamara,7212


In [ ]:
# @title
# === Cell 5 - Extract main-output prestation codes & create HAS_map_main.csv ===

from pathlib import Path
from IPython.display import display

try:
    data_df = df_out.copy()
except NameError:
    print("Error: 'df_out' not found. Attempting to reload adjusted output...")
    if 'OUTPUT_FILE' in locals():
        original_ext = Path(OUTPUT_FILE).suffix.lower()
        if original_ext == ".xlsx":
            data_df = pd.read_excel(OUTPUT_FILE)
        else:
            data_df = pd.read_csv(OUTPUT_FILE)
        print(f"Reloaded adjusted main output from {OUTPUT_FILE}")
    else:
        raise NameError("Neither 'df_out' nor 'OUTPUT_FILE' is defined. Cannot proceed.")

CODE_COL_CANDIDATES = ["N\u00b0 Prestation", "No prestation", "Code prestation", "Code", "Prestation"]
code_col_map = next((c for c in CODE_COL_CANDIDATES if c in data_df.columns), None)
if code_col_map is None:
    raise ValueError(f"Could not find prestation code column; tried {CODE_COL_CANDIDATES}")

if "norm_code" in globals():
    codes = data_df[code_col_map].apply(norm_code)
else:
    codes = data_df[code_col_map].astype(str).str.strip()

unique_prestation_codes = sorted([c for c in codes.dropna().astype(str).unique().tolist() if c])

print(f"\nUnique main-output '{code_col_map}' codes found:")
unique_codes_df = pd.DataFrame(unique_prestation_codes, columns=['Unique Prestation Codes'])
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(unique_codes_df)
print(f"\nTotal unique main-output codes: {len(unique_prestation_codes)}")

prestation_map_df = pd.DataFrame({
    'Code_ext': unique_prestation_codes,
    'Leistungstarif_nummer': unique_prestation_codes,
})

output_csv_path = Path(root_folder) / "HAS_map_main.csv"
prestation_map_df.to_csv(output_csv_path, index=False, sep=';')

print(f"\nCreated and saved main-output prestation map to:\n   {output_csv_path}")
print("\nFirst 10 rows of the generated main prestation map:")
display(prestation_map_df.head(10))



Unique main-output 'No prestation' codes found:


,Unique Prestation Codes
0,11000
1,11100
2,11200
3,14100
4,14200
5,15104
6,15204
7,16009
8,16011
9,50020



Total unique main-output codes: 17

Created and saved main-output prestation map to:
   /content/drive/MyDrive/NE_15minsreduit_April2026/HAS_map_main.csv

First 10 rows of the generated main prestation map:


,Code_ext,Leistungstarif_nummer
0,11000,11000
1,11100,11100
2,11200,11200
3,14100,14100
4,14200,14200
5,15104,15104
6,15204,15204
7,16009,16009
8,16011,16011
9,50020,50020


In [ ]:
# @title Cell 5B - Whitelisted 201 -> 101 transfer export

import re
import unicodedata
from pathlib import Path
from IPython.display import display

try:
    df_whitelist
    root_folder
except NameError:
    raise NameError("Run the whitelist fork/reduction cell and export-root setup before this transfer cell.")

TRANSFER_FOLDER_NAME = "02_Whitelisted_Ready_For_101"
TRANSFER_TARGET_OE_VALUE = globals().get("TRANSFER_TARGET_OE_VALUE", "100000000000000101")

def norm_col_transfer(s):
    text = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    text = text.strip().lower().replace("#", "no")
    text = re.sub(r"n\s*o", "no", text)
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")

def pick_col_transfer(df_, candidates):
    lookup = {norm_col_transfer(c): c for c in df_.columns}
    for cand in candidates:
        key = norm_col_transfer(cand)
        if key in lookup:
            return lookup[key]
    return None

def format_date_transfer(d):
    dt = pd.to_datetime(d, dayfirst=True, errors="coerce")
    return "" if pd.isna(dt) else dt.strftime("%d.%m.%Y")

def format_time_transfer(t):
    if pd.isna(t):
        return ""
    if isinstance(t, pd.Timestamp):
        return t.strftime("%H:%M")
    try:
        dt = pd.to_datetime(t, errors="raise", dayfirst=True)
        return dt.strftime("%H:%M")
    except Exception:
        pass
    if isinstance(t, (float, int)) and not isinstance(t, bool):
        hours, remainder = divmod(float(t) * 24, 1)
        minutes = int(round(remainder * 60))
        return f"{int(hours):02d}:{minutes:02d}"
    s = str(t).strip()
    tok = s.split()[-1]
    parts = tok.split(":")
    if len(parts) >= 2 and parts[0].lstrip("+-").isdigit() and parts[1].isdigit():
        return f"{int(parts[0]):02d}:{int(parts[1]):02d}"
    return s[:5]

def numeric_id_series(s):
    extracted = s.astype(str).str.extract(r"(\d+)")[0]
    return pd.to_numeric(extracted, errors="coerce").astype("Int64")

def normalize_code_transfer(x):
    if "norm_code" in globals():
        return norm_code(x)
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    runs = re.findall(r"\d+", s)
    return sorted(runs, key=len, reverse=True)[0] if runs else s

def explicit_or_auto_col(df_map, explicit_name, role, side):
    if explicit_name and str(explicit_name).strip():
        wanted = norm_col_transfer(explicit_name)
        lookup = {norm_col_transfer(c): c for c in df_map.columns}
        if wanted not in lookup:
            raise ValueError(f"Configured mapping column '{explicit_name}' not found. Available: {list(df_map.columns)}")
        return lookup[wanted]

    role_tokens = {
        "client": ["client", "klient", "kunde", "kd"],
        "collab": ["collab", "collaborateur", "collaborator", "mitarbeiter", "employee"],
    }[role]
    entity_token = "101" if side == "target" else "201"
    side_tokens = ["sa", "target", "to", "new", "101"] if side == "target" else ["sarl", "source", "from", "old", "201"]

    scored = []
    for col in df_map.columns:
        n = norm_col_transfer(col)
        if not any(tok in n for tok in role_tokens):
            continue
        score = 0
        if entity_token in n:
            score += 10
        if any(tok in n for tok in side_tokens):
            score += 3
        if any(tok in n for tok in ["no", "nr", "id", "nummer"]):
            score += 1
        if score > 0:
            scored.append((score, col, n))

    if not scored:
        raise ValueError(
            f"Could not auto-detect {side} {role} mapping column. "
            f"Set the matching *_MAP_COL variable. Available: {list(df_map.columns)}"
        )
    scored.sort(key=lambda x: (-x[0], x[2]))
    best_score = scored[0][0]
    best = [col for score, col, _ in scored if score == best_score]
    if len(best) > 1:
        raise ValueError(f"Ambiguous {side} {role} mapping columns: {best}. Set the matching *_MAP_COL variable explicitly.")
    return best[0]

def write_transfer_batch(batch_path: Path, csv_name: str):
    batch_text = (
        "@echo off\n"
        "chcp 65001\n"
        f"\"..\\nx-spi-client\\Asebis.Client.StarterCommand.exe\" "
        f"/u=nexus /p=fAvNCDnW3E /t=ImportLeistungen_CSV /o={TRANSFER_TARGET_OE_VALUE} "
        f"/f=\"{csv_name}\" /map=\"..\\HAS_map.csv\" /v\n"
        "Pause\n"
    )
    with open(batch_path, "w", encoding="utf-8") as f:
        f.write(batch_text)

folder_whitelist = Path(root_folder) / TRANSFER_FOLDER_NAME
folder_whitelist.mkdir(parents=True, exist_ok=True)

if df_whitelist.empty:
    print("No whitelisted rows found. Created transfer folder but skipped mapping and CSV generation:")
    print(" ", folder_whitelist)
    df_whitelist_mapped = df_whitelist.copy()
    whitelist_transfer_csv_df = pd.DataFrame()
else:
    map_path = Path(resolve_mapping_workbook())
    xls = pd.ExcelFile(map_path)
    clients_sheet = next((s for s in xls.sheet_names if "client" in norm_col_transfer(s)), None)
    collabs_sheet = next((s for s in xls.sheet_names if "collab" in norm_col_transfer(s) or "collabor" in norm_col_transfer(s)), None)
    if not clients_sheet or not collabs_sheet:
        raise ValueError(f"Could not find both clients and collabs sheets in {map_path}. Found: {xls.sheet_names}")

    clients_map_df = pd.read_excel(map_path, sheet_name=clients_sheet)
    collabs_map_df = pd.read_excel(map_path, sheet_name=collabs_sheet)

    src_client_col = explicit_or_auto_col(clients_map_df, globals().get("SOURCE_CLIENT_MAP_COL", ""), "client", "source")
    tgt_client_col = explicit_or_auto_col(clients_map_df, globals().get("TARGET_CLIENT_MAP_COL", ""), "client", "target")
    src_collab_col = explicit_or_auto_col(collabs_map_df, globals().get("SOURCE_COLLAB_MAP_COL", ""), "collab", "source")
    tgt_collab_col = explicit_or_auto_col(collabs_map_df, globals().get("TARGET_COLLAB_MAP_COL", ""), "collab", "target")

    print("Mapping workbook:", map_path)
    print("Clients sheet:", clients_sheet, "|", src_client_col, "->", tgt_client_col)
    print("Collabs sheet:", collabs_sheet, "|", src_collab_col, "->", tgt_collab_col)

    client_map = {
        int(k): int(v)
        for k, v in zip(numeric_id_series(clients_map_df[src_client_col]), numeric_id_series(clients_map_df[tgt_client_col]))
        if pd.notna(k) and pd.notna(v)
    }
    collab_map = {
        int(k): int(v)
        for k, v in zip(numeric_id_series(collabs_map_df[src_collab_col]), numeric_id_series(collabs_map_df[tgt_collab_col]))
        if pd.notna(k) and pd.notna(v)
    }

    date_col_transfer = pick_col_transfer(df_whitelist, ["Date D\u00e9but", "Date", "Jour", "Date de prestation"])
    start_col_transfer = pick_col_transfer(df_whitelist, ["D\u00e9but", "Heure d\u00e9but", "Heure Debut", "Von"])
    end_col_transfer = pick_col_transfer(df_whitelist, ["Fin", "Heure fin", "Bis"])
    prest_col_transfer = pick_col_transfer(df_whitelist, ["N\u00b0 Prestation", "No prestation", "Code prestation", "Prestation", "Code"])
    duree_col_transfer = pick_col_transfer(df_whitelist, ["Dur\u00e9e", "Duree", "Dur\u00e9e (min)", "Dur\u00e9e (minutes)", "Dauer_verrechnet"])
    client_col_transfer = pick_col_transfer(df_whitelist, ["N\u00b0 du client", "No client", "ID client", "Client", "KD-Nr", "KD_Nr"])
    collab_col_transfer = pick_col_transfer(df_whitelist, ["No collaborateur", "Collaborateur", "ID collaborateur", "Mitarbeiter-ID"])
    required_transfer_cols = [
        ("date", date_col_transfer), ("start", start_col_transfer), ("end", end_col_transfer),
        ("prestation", prest_col_transfer), ("duration", duree_col_transfer),
        ("client", client_col_transfer), ("collaborator", collab_col_transfer),
    ]
    missing_transfer_cols = [name for name, col in required_transfer_cols if col is None]
    if missing_transfer_cols:
        raise ValueError(f"Missing required whitelist transfer columns: {missing_transfer_cols}")

    df_whitelist_mapped = df_whitelist.copy()
    df_whitelist_mapped["orig_client_no"] = numeric_id_series(df_whitelist_mapped[client_col_transfer])
    df_whitelist_mapped["orig_collab_no"] = numeric_id_series(df_whitelist_mapped[collab_col_transfer])
    df_whitelist_mapped["KD-Nr"] = df_whitelist_mapped["orig_client_no"].map(client_map).fillna(0).astype(int)
    df_whitelist_mapped["Mitarbeiter-ID"] = df_whitelist_mapped["orig_collab_no"].map(collab_map).fillna(0).astype(int)
    df_whitelist_mapped["Einsatzgrund"] = df_whitelist_mapped["KD-Nr"].apply(lambda x: 2 if int(x) != 0 else 0).astype(int)

    whitelist_transfer_csv_df = pd.DataFrame({
        "Datum": df_whitelist_mapped[date_col_transfer].apply(format_date_transfer),
        "Von": df_whitelist_mapped[start_col_transfer].apply(format_time_transfer),
        "Bis": df_whitelist_mapped[end_col_transfer].apply(format_time_transfer),
        "Leistungscode": df_whitelist_mapped[prest_col_transfer].apply(normalize_code_transfer).astype(str),
        "Dauer_verrechnet": pd.to_numeric(df_whitelist_mapped[duree_col_transfer], errors="coerce").fillna(0).astype(int),
        "OE": TRANSFER_TARGET_OE_VALUE,
        "KD-Nr": df_whitelist_mapped["KD-Nr"].fillna(0).astype(int),
        "Klient": 0,
        "Einsatzgrund": df_whitelist_mapped["Einsatzgrund"].fillna(0).astype(int),
        "Mitarbeiter-ID": df_whitelist_mapped["Mitarbeiter-ID"].fillna(0).astype(int),
    })

    unique_wl_codes = sorted([c for c in whitelist_transfer_csv_df["Leistungscode"].dropna().astype(str).unique().tolist() if c])
    whitelist_map_df = pd.DataFrame({"Code_ext": unique_wl_codes, "Leistungstarif_nummer": unique_wl_codes})
    whitelist_map_path = Path(root_folder) / "HAS_map.csv"
    whitelist_map_df.to_csv(whitelist_map_path, index=False, sep=";")

    total_wl = int(whitelist_transfer_csv_df["Dauer_verrechnet"].fillna(0).sum())
    transfer_csv_name = f"RDA_Whitelisted_Ready_For_101+{total_wl}.csv"
    transfer_csv_path = folder_whitelist / transfer_csv_name
    whitelist_transfer_csv_df.to_csv(transfer_csv_path, index=False, sep=";", encoding="utf-8")

    transfer_batch_path = folder_whitelist / "RDA_Whitelisted_Ready_For_101_batch.bat"
    write_transfer_batch(transfer_batch_path, transfer_csv_name)

    mask_unmapped_client = df_whitelist_mapped["orig_client_no"].notna() & (df_whitelist_mapped["orig_client_no"] != 0) & (df_whitelist_mapped["KD-Nr"] == 0)
    mask_unmapped_collab = df_whitelist_mapped["orig_collab_no"].notna() & (df_whitelist_mapped["orig_collab_no"] != 0) & (df_whitelist_mapped["Mitarbeiter-ID"] == 0)
    whitelist_mapping_summary_df = pd.DataFrame([
        ["Whitelisted rows", len(df_whitelist_mapped)],
        ["Total whitelist minutes", total_wl],
        ["Mapped client rows", int((df_whitelist_mapped["KD-Nr"] != 0).sum())],
        ["Unmapped client rows", int(mask_unmapped_client.sum())],
        ["Mapped collaborator rows", int((df_whitelist_mapped["Mitarbeiter-ID"] != 0).sum())],
        ["Unmapped collaborator rows", int(mask_unmapped_collab.sum())],
        ["HAS map", str(whitelist_map_path)],
        ["CSV", str(transfer_csv_path)],
        ["Batch", str(transfer_batch_path)],
    ], columns=["Metric", "Value"])

    qa_path = folder_whitelist / "RDA_Whitelisted_Ready_For_101_QA.xlsx"
    with pd.ExcelWriter(qa_path, engine="openpyxl") as writer:
        whitelist_mapping_summary_df.to_excel(writer, index=False, sheet_name="Summary")
        whitelist_transfer_csv_df.to_excel(writer, index=False, sheet_name="Import_CSV")
        df_whitelist_mapped.to_excel(writer, index=False, sheet_name="Mapped_Source_Rows")

    print("\nWhitelist transfer package created:")
    print("  Folder:", folder_whitelist)
    print("  CSV:", transfer_csv_path)
    print("  BAT:", transfer_batch_path)
    print("  HAS map:", whitelist_map_path)
    print("  QA:", qa_path)
    display(whitelist_mapping_summary_df)
    display(whitelist_transfer_csv_df.head(20))


No whitelisted rows found. Created transfer folder but skipped mapping and CSV generation:
  /content/drive/MyDrive/NE_15minsreduit_April2026/02_Whitelisted_Ready_For_101


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# @title Cell 5C - Export Whitelisted Rows in Original RDA Format

try:
    df_whitelist
    folder_whitelist
except NameError:
    raise NameError("Please run Cell 2 (Whitelist Fork) and Cell 5B (Transfer export) first.")

# Define the output filename and path
output_filename = "RDA_Whitelisted_Original_Format.xlsx"
original_format_export_path = Path(folder_whitelist) / output_filename

if not df_whitelist.empty:
    # Export the unmapped, raw whitelisted DataFrame to Excel
    df_whitelist.to_excel(original_format_export_path, index=False)
    print(f"\u2705 Exported whitelisted RDAs in original format to:\n   {original_format_export_path}")
    print(f"\nPreview of the original format rows ({len(df_whitelist)} total):")
    display(df_whitelist.head(10))
else:
    print("\u26a0\ufe0f No whitelisted rows to export. (df_whitelist is empty)")


⚠️ No whitelisted rows to export. (df_whitelist is empty)


In [ ]:
# @title
# === Cell 6 — Dashboard récap des contrôles / comparisons ===

from IPython.display import display

# --- Recover original & adjusted DataFrames ---
try:
    if "df_main_raw" in globals():
        original_df = df_main_raw.copy()
        print("Dashboard reference: non-whitelisted raw rows only.")
    else:
        original_df = df.copy()
except NameError:
    if 'input_path' in locals():
        ext = input_path.suffix.lower()
        if ext == ".xlsx":
            original_df = pd.read_excel(input_path)
        else:
            original_df = pd.read_csv(input_path)
        print(f"Reloaded original_df from {input_path}")
    else:
        raise NameError("Cannot find original df.")

try:
    adjusted_df = df_out.copy()
except NameError:
    if 'OUTPUT_FILE' in locals():
        ext = Path(OUTPUT_FILE).suffix.lower()
        if ext == ".xlsx":
            adjusted_df = pd.read_excel(OUTPUT_FILE)
        else:
            adjusted_df = pd.read_csv(OUTPUT_FILE)
        print(f"Reloaded adjusted_df from {OUTPUT_FILE}")
    else:
        raise NameError("Cannot find adjusted df_out / OUTPUT_FILE.")

# --- Detect duree_col & collab_col if needed ---
try:
    duree_col
except NameError:
    DUREE_COL_CANDIDATES = ['Durée', 'Duree', 'Durée (min)', 'Durée (minutes)']
    duree_col = next((c for c in DUREE_COL_CANDIDATES if c in original_df.columns), None)
    if duree_col is None:
        raise ValueError(f"Could not find a duration column; tried {DUREE_COL_CANDIDATES}")

try:
    collab_col
except NameError:
    COLLAB_COL_CANDIDATES  = ['No collaborateur', 'Collaborateur', 'ID collaborateur']
    collab_col = next((c for c in COLLAB_COL_CANDIDATES if c in original_df.columns), None)
    if collab_col is None:
        raise ValueError(f"Could not find a collaborator column; tried {COLLAB_COL_CANDIDATES}")

print("===== DASHBOARD RÉCAP =====\n")

# 1) Global total durations (original vs adjusted)
total_duration_original = original_df[duree_col].fillna(0).sum()
total_duration_adjusted = adjusted_df[duree_col].fillna(0).sum()

print(f"1) Global '{duree_col}' (minutes) — original vs adjusted")
print(f"   Original total : {total_duration_original}")
print(f"   Adjusted total : {total_duration_adjusted}")
print(f"   Diff (Orig - Adj): {total_duration_original - total_duration_adjusted}")

if np.isclose(total_duration_original, total_duration_adjusted):
    print("   ✅ Totaux globaux Durée : MATCH\n")
else:
    print("   ⚠️ Totaux globaux Durée : MISMATCH\n")

# 2) Temps retiré / Temps ajouté totals
if 'Temps retiré' in adjusted_df.columns and 'Temps ajouté' in adjusted_df.columns:
    total_retire = adjusted_df['Temps retiré'].fillna(0).sum()
    total_ajoute = adjusted_df['Temps ajouté'].fillna(0).sum()

    print("2) Totaux 'Temps retiré' vs 'Temps ajouté' (minutes)")
    print(f"   Total Temps retiré : {total_retire}")
    print(f"   Total Temps ajouté : {total_ajoute}")
    print(f"   Diff (Retiré - Ajouté): {total_retire - total_ajoute}")

    if np.isclose(total_retire, total_ajoute):
        print("   ✅ Temps retiré == Temps ajouté\n")
    else:
        print("   ⚠️ Temps retiré != Temps ajouté\n")
else:
    print("2) Temps retiré / Temps ajouté: colonnes manquantes.\n")

# 3) Original Durée vs derived _dur (from Cell 2) on original df
if '_dur' in df.columns:
    total_duree_original_col = df[duree_col].fillna(0).sum()
    total_duree_derived = df['_dur'].fillna(0).sum()
    diff_orig_vs_derived = total_duree_original_col - total_duree_derived

    print("3) Check original Durée vs _dur (calculé depuis Début/Fin, fichier brut)")
    print(f"   Total Durée (colonne) : {total_duree_original_col}")
    print(f"   Total _dur (brut)     : {total_duree_derived}")
    print(f"   Diff (Durée - _dur)   : {diff_orig_vs_derived}")
    if np.isclose(total_duree_original_col, total_duree_derived):
        print("   ✅ Durée et _dur (brut) cohérents\n")
    else:
        print("   ⚠️ Écart entre Durée et _dur (brut)\n")
else:
    print("3) Colonne '_dur' absente dans df (peut-être cell 2 non exécutée dans cette session).\n")

# 4) Per-collaborateur comparison (original vs adjusted)
print("4) Différences par collaborateur (Durée totale)")

original_collab_duration = original_df.groupby(collab_col)[duree_col].sum().reset_index()
original_collab_duration.rename(columns={duree_col: 'Original_Durée_Total'}, inplace=True)

adjusted_collab_duration = adjusted_df.groupby(collab_col)[duree_col].sum().reset_index()
adjusted_collab_duration.rename(columns={duree_col: 'Adjusted_Durée_Total'}, inplace=True)

comparison_df = pd.merge(
    original_collab_duration,
    adjusted_collab_duration,
    on=collab_col,
    how='outer'
).fillna(0)

comparison_df['Difference (Original - Adjusted)'] = (
    comparison_df['Original_Durée_Total'] - comparison_df['Adjusted_Durée_Total']
)

# Store globally to inspect all rows if needed
collab_duration_comparison_df = comparison_df.copy()

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(comparison_df.sort_values(by='Difference (Original - Adjusted)', ascending=False).head(30))

total_diff = comparison_df['Difference (Original - Adjusted)'].sum()
if np.isclose(total_diff, 0):
    print(f"\n   ✅ Somme des différences par collab = 0 (cohérent avec global)")
else:
    print(f"\n   ⚠️ Somme des différences par collab = {total_diff} (vérifier)")

major_discrepancies = comparison_df[comparison_df['Difference (Original - Adjusted)'].abs() > 0.001]
if not major_discrepancies.empty:
    print("\n   Collaborateurs avec différences (toutes non-nulles):")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(major_discrepancies.sort_values(by='Difference (Original - Adjusted)', ascending=False))
else:
    print("\n   ✅ Aucun collaborateur avec différence notable après ajustement\n")

# 5) Overlap summary
print("5) Overlaps (après ajustement, même collab + même jour)")
if 'overlaps_df' in globals() and not overlaps_df.empty:
    print(f"   ⚠️ Nombre de paires en overlap : {len(overlaps_df)}")
    display(overlaps_df.head(20))
else:
    print("   ✅ Aucun overlap détecté (selon Cell 3)\n")

print("===== FIN DU DASHBOARD =====")


Dashboard reference: non-whitelisted raw rows only.
===== DASHBOARD RÉCAP =====

1) Global 'Durée' (minutes) — original vs adjusted
   Original total : 60216
   Adjusted total : 60216
   Diff (Orig - Adj): 0
   ✅ Totaux globaux Durée : MATCH

2) Totaux 'Temps retiré' vs 'Temps ajouté' (minutes)
   Total Temps retiré : 4350
   Total Temps ajouté : 4350
   Diff (Retiré - Ajouté): 0
   ✅ Temps retiré == Temps ajouté

3) Colonne '_dur' absente dans df (peut-être cell 2 non exécutée dans cette session).

4) Différences par collaborateur (Durée totale)


,No collaborateur,Original_Durée_Total,Adjusted_Durée_Total,Difference (Original - Adjusted)
0,1209,7234,7234,0
1,2004,4792,4792,0
2,2008,6979,6979,0
3,2012,7732,7732,0
4,2014,7212,7212,0
5,2016,660,660,0
6,2017,7266,7266,0
7,2020,1836,1836,0
8,2023,224,224,0
9,2026,5947,5947,0



   ✅ Somme des différences par collab = 0 (cohérent avec global)

   ✅ Aucun collaborateur avec différence notable après ajustement

5) Overlaps (après ajustement, même collab + même jour)
   ✅ Aucun overlap détecté (selon Cell 3)

===== FIN DU DASHBOARD =====


In [ ]:
# @title
collab_id_col = 'No collaborateur'
collab_name_col = 'Collaborateur'

# Select the two columns and get unique pairs
all_collabs_df = adjusted_df[[collab_id_col, collab_name_col]].drop_duplicates().reset_index(drop=True)

print(f"Extracted {len(all_collabs_df)} unique collaborator pairs:")
display(all_collabs_df.head(10))

Extracted 13 unique collaborator pairs:


,No collaborateur,Collaborateur
0,1209,Sihyürek Mine
1,2004,Incerti Mélanie
2,2008,Le Pavec Anna
3,2012,Belbahri Hakima
4,2014,Mojsilovic Tamara
5,2016,Tissot Katia
6,2017,Sbarzella Andrea
7,2020,Hafidi Moustapha
8,2023,Sutterlet Pascal
9,2026,Thévoz Martine


In [ ]:
# @title
collabs_61010_df = adjusted_df[adjusted_df[prest_col] == '61010'][[collab_id_col, collab_name_col]].drop_duplicates().reset_index(drop=True)

print(f"Extracted {len(collabs_61010_df)} unique collaborator pairs with prestation 61010:")
display(collabs_61010_df.head(10))

Extracted 10 unique collaborator pairs with prestation 61010:


,No collaborateur,Collaborateur
0,1209,Sihyürek Mine
1,2004,Incerti Mélanie
2,2008,Le Pavec Anna
3,2012,Belbahri Hakima
4,2014,Mojsilovic Tamara
5,2017,Sbarzella Andrea
6,2020,Hafidi Moustapha
7,2026,Thévoz Martine
8,2027,Soler Fanny
9,2029,Staudacher Joane


In [ ]:
# @title
client_col = 'N° du client'
all_clients_df = pd.DataFrame(adjusted_df[client_col].unique(), columns=[client_col])

print(f"Extracted {len(all_clients_df)} unique clients:")
display(all_clients_df.head(10))

Extracted 65 unique clients:


,N° du client
0,NaN
1,202863.0
2,104115.0
3,202896.0
4,202940.0
5,103830.0
6,202997.0
7,202939.0
8,103787.0
9,103846.0


In [ ]:
# @title
import sys
!{sys.executable} -m pip install xlsxwriter

output_excel_path = Path(root_folder) / "Collaborator_Client_Summary.xlsx"

with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:
    all_collabs_df.to_excel(writer, sheet_name='All Collaborators', index=False)
    collabs_61010_df.to_excel(writer, sheet_name='61010 Collaborators', index=False)
    all_clients_df.to_excel(writer, sheet_name='All Clients', index=False)

print(f"\n✅ Successfully created Excel file with collaborator and client summaries:\n   {output_excel_path}")


✅ Successfully created Excel file with collaborator and client summaries:
   /content/drive/MyDrive/NE_15minsreduit_April2026/Collaborator_Client_Summary.xlsx


In [ ]:
# @title ✅ FINAL AUDIT — Check duration consistency + sums per output folder

import pandas as pd
import numpy as np
import datetime as dt
from pathlib import Path
from IPython.display import display

print("===== FINAL AUDIT: DURATIONS + OUTPUT FOLDER SUMS =====\n")

# ============================================================
# Helpers
# ============================================================

def pick_col(df_, candidates):
    return next((c for c in candidates if c in df_.columns), None)

def to_min_audit(t):
    """
    Converts time to minutes since midnight.
    Handles:
    - pandas Timestamp
    - datetime.time
    - Excel time floats
    - strings like '08:30', '08:30:00'
    """
    if pd.isna(t):
        return np.nan

    if isinstance(t, pd.Timestamp):
        return int(t.hour) * 60 + int(t.minute)

    if isinstance(t, dt.time):
        return int(t.hour) * 60 + int(t.minute)

    if isinstance(t, (float, int)) and not isinstance(t, bool):
        x = float(t)

        # Excel fraction of day
        if 0 <= x < 1.0:
            return int(round(x * 24 * 60)) % 1440

        # minutes since midnight
        if 0 <= x < 1440 and abs(x - round(x)) < 1e-9:
            return int(round(x))

        # decimal hours
        if 0 <= x < 24:
            return int(round(x * 60))

    s = str(t).strip()

    parsed = pd.to_datetime(s, dayfirst=True, errors="coerce")
    if not pd.isna(parsed):
        return int(parsed.hour) * 60 + int(parsed.minute)

    tok = s.split()[-1]
    parts = tok.split(":")
    if len(parts) >= 2:
        return int(parts[0]) * 60 + int(parts[1])

    return np.nan

def duration_from_start_end(start_min, end_min):
    """
    Calculates duration from start/end.
    If end < start, assumes crossing midnight.
    """
    if pd.isna(start_min) or pd.isna(end_min):
        return np.nan

    start_min = int(start_min)
    end_min = int(end_min)

    diff = end_min - start_min
    return diff if diff >= 0 else diff + 1440

def audit_duration_table(df_, start_col, end_col, duration_col, label, extra_cols=None, show_bad=True):
    """
    Checks every row where duration column equals calculated End - Start.
    Returns summary + bad rows.
    """
    check = df_.copy()

    check["_start_min_audit"] = check[start_col].apply(to_min_audit)
    check["_end_min_audit"] = check[end_col].apply(to_min_audit)

    check["_duration_from_times"] = check.apply(
        lambda r: duration_from_start_end(r["_start_min_audit"], r["_end_min_audit"]),
        axis=1
    )

    check["_duration_col"] = pd.to_numeric(check[duration_col], errors="coerce")
    check["_delta_duration"] = check["_duration_col"] - check["_duration_from_times"]

    total_duration_col = check["_duration_col"].fillna(0).sum()
    total_duration_from_times = check["_duration_from_times"].fillna(0).sum()
    total_delta = total_duration_col - total_duration_from_times

    bad = check[check["_delta_duration"].fillna(0) != 0].copy()

    print(f"--- {label} ---")
    print(f"Rows checked                 : {len(check):,}")
    print(f"Sum duration column          : {int(total_duration_col)}")
    print(f"Sum calculated from times    : {int(total_duration_from_times)}")
    print(f"Difference                   : {int(total_delta)}")
    print(f"Rows with duration mismatch  : {len(bad):,}")

    if bad.empty:
        print("✅ OK: duration matches start/end.\n")
    else:
        print("⚠️ MISMATCH FOUND.\n")

        if show_bad:
            cols = []
            if extra_cols:
                cols.extend([c for c in extra_cols if c in bad.columns])

            cols.extend([
                start_col,
                end_col,
                duration_col,
                "_duration_from_times",
                "_delta_duration"
            ])

            display(bad[cols].head(50))

    return {
        "label": label,
        "rows": len(check),
        "sum_duration_col": int(total_duration_col),
        "sum_duration_from_times": int(total_duration_from_times),
        "difference": int(total_delta),
        "mismatch_rows": len(bad),
        "bad_rows": bad
    }

# ============================================================
# 1) Audit df_out
# ============================================================

if "df_out" not in globals():
    raise NameError("df_out not found. Run Cell 2 first.")

adjusted_df = df_out.copy()

DATE_COL_CANDIDATES = ["Date Début", "Date", "Jour", "Date de prestation"]
COLLAB_COL_CANDIDATES = ["No collaborateur", "Collaborateur", "ID collaborateur"]
CODE_COL_CANDIDATES = ["N° Prestation", "No prestation", "Code prestation", "Prestation", "Code"]
DUREE_COL_CANDIDATES = ["Durée", "Duree", "Durée (min)", "Durée (minutes)"]

date_col_audit = pick_col(adjusted_df, DATE_COL_CANDIDATES)
collab_col_audit = pick_col(adjusted_df, COLLAB_COL_CANDIDATES)
code_col_audit = pick_col(adjusted_df, CODE_COL_CANDIDATES)
duree_col_audit = pick_col(adjusted_df, DUREE_COL_CANDIDATES)

if duree_col_audit is None:
    raise ValueError("Could not find duration column in df_out.")

df_out_result = audit_duration_table(
    adjusted_df,
    start_col="Début",
    end_col="Fin",
    duration_col=duree_col_audit,
    label="Adjusted df_out",
    extra_cols=[date_col_audit, collab_col_audit, code_col_audit, "Temps retiré", "Temps ajouté"]
)

df_out_total_duration = df_out_result["sum_duration_col"]

# ============================================================
# 2) Audit generated Nexus CSVs + sum by main output folder
# ============================================================

if "root_folder" not in globals():
    raise NameError("root_folder not found. Run the export cell first.")

root = Path(root_folder)

if not root.exists():
    raise FileNotFoundError(f"root_folder does not exist: {root}")

MAIN_OUTPUT_FOLDERS = [
    "01_All_Collabs_One_CSV",
    "02_Collabs_With_61010_One_CSV",
    "03_Per_Collab_Separate",
    "02_Whitelisted_Ready_For_101"
]

folder_results = []
csv_results = []
all_bad_rows = []

print("===== GENERATED CSV AUDIT =====\n")

for folder_name in MAIN_OUTPUT_FOLDERS:
    folder_path = root / folder_name

    if not folder_path.exists():
        folder_results.append({
            "Folder": folder_name,
            "Folder exists": False,
            "CSV files": 0,
            "Rows": 0,
            "Sum Dauer_verrechnet": 0,
            "Sum calculated Von/Bis": 0,
            "Difference": 0,
            "Mismatch rows": 0
        })
        continue

    csv_files = sorted(folder_path.rglob("*.csv"))

    folder_csv_count = 0
    folder_rows = 0
    folder_duration_sum = 0
    folder_time_sum = 0
    folder_mismatch_rows = 0

    for csv_path in csv_files:
        try:
            csv_df = pd.read_csv(csv_path, sep=";")
        except Exception as e:
            print(f"⚠️ Could not read CSV: {csv_path}")
            print(e)
            continue

        # Only audit Nexus import CSVs
        required_cols = {"Von", "Bis", "Dauer_verrechnet"}
        if not required_cols.issubset(csv_df.columns):
            continue

        folder_csv_count += 1

        result = audit_duration_table(
            csv_df,
            start_col="Von",
            end_col="Bis",
            duration_col="Dauer_verrechnet",
            label=f"{folder_name} / {csv_path.name}",
            extra_cols=["Datum", "Mitarbeiter-ID", "Leistungscode", "KD-Nr"],
            show_bad=False
        )

        folder_rows += result["rows"]
        folder_duration_sum += result["sum_duration_col"]
        folder_time_sum += result["sum_duration_from_times"]
        folder_mismatch_rows += result["mismatch_rows"]

        csv_results.append({
            "Folder": folder_name,
            "CSV file": str(csv_path.relative_to(root)),
            "Rows": result["rows"],
            "Sum Dauer_verrechnet": result["sum_duration_col"],
            "Sum calculated Von/Bis": result["sum_duration_from_times"],
            "Difference": result["difference"],
            "Mismatch rows": result["mismatch_rows"]
        })

        if result["mismatch_rows"] > 0:
            bad = result["bad_rows"].copy()
            bad["_file"] = str(csv_path.relative_to(root))
            all_bad_rows.append(bad)

    folder_results.append({
        "Folder": folder_name,
        "Folder exists": True,
        "CSV files": folder_csv_count,
        "Rows": folder_rows,
        "Sum Dauer_verrechnet": int(folder_duration_sum),
        "Sum calculated Von/Bis": int(folder_time_sum),
        "Difference": int(folder_duration_sum - folder_time_sum),
        "Mismatch rows": int(folder_mismatch_rows)
    })

# ============================================================
# 3) Display folder summary
# ============================================================

folder_summary_df = pd.DataFrame(folder_results)

print("\n===== SUM OF DURATION BY MAIN OUTPUT FOLDER =====")
display(folder_summary_df)

print("\nReference:")
print(f"df_out total duration: {df_out_total_duration}")

print("\nExpected logic:")
print("01_All_Collabs_One_CSV should usually equal df_out total.")
print("03_Per_Collab_Separate should usually equal df_out total.")
print("02_Collabs_With_61010_One_CSV will be smaller unless every collaborator has 61010.")

# ============================================================
# 4) Display per-file summary
# ============================================================

csv_summary_df = pd.DataFrame(csv_results)

if not csv_summary_df.empty:
    print("\n===== PER CSV FILE SUMMARY =====")
    display(csv_summary_df)
else:
    print("\n⚠️ No generated Nexus CSV files found in the 3 main folders.")

# ============================================================
# 5) Show bad rows if any
# ============================================================

if all_bad_rows:
    all_bad_rows_df = pd.concat(all_bad_rows, ignore_index=True)
    print("\n⚠️ Rows where Dauer_verrechnet does NOT match Von/Bis:")
    display(all_bad_rows_df.head(100))
else:
    print("\n✅ All generated Nexus CSV rows have Dauer_verrechnet matching Von/Bis.")

# ============================================================
# 6) Split total check: main output + whitelist transfer
# ============================================================

main_raw_total = 0
whitelist_raw_total = 0
if "df_main_raw" in globals():
    main_duration_col = pick_col(df_main_raw, DUREE_COL_CANDIDATES)
    if main_duration_col:
        main_raw_total = int(pd.to_numeric(df_main_raw[main_duration_col], errors="coerce").fillna(0).sum())
if "df_whitelist" in globals():
    wl_duration_col = pick_col(df_whitelist, DUREE_COL_CANDIDATES)
    if wl_duration_col:
        whitelist_raw_total = int(pd.to_numeric(df_whitelist[wl_duration_col], errors="coerce").fillna(0).sum())

transfer_total = int(
    folder_summary_df.loc[
        folder_summary_df["Folder"] == "02_Whitelisted_Ready_For_101",
        "Sum Dauer_verrechnet"
    ].sum()
)
split_raw_total = main_raw_total + whitelist_raw_total
combined_export_total = df_out_total_duration + transfer_total

print("\n===== SPLIT TOTAL CHECK =====")
print(f"Main raw duration after whitelist removal : {main_raw_total}")
print(f"Whitelisted raw duration                 : {whitelist_raw_total}")
print(f"Raw split total                          : {split_raw_total}")
print(f"Adjusted main df_out duration            : {df_out_total_duration}")
print(f"Whitelist transfer CSV duration          : {transfer_total}")
print(f"Combined export duration                 : {combined_export_total}")

split_total_ok = combined_export_total == split_raw_total
if split_total_ok:
    print("OK: main adjusted duration + whitelist transfer duration equals the split raw total.")
else:
    print("WARNING: combined export duration does not equal the split raw total.")

# ============================================================
# 7) Final verdict
# ============================================================

print("\n===== FINAL VERDICT =====")

df_out_ok = df_out_result["mismatch_rows"] == 0
csv_ok = folder_summary_df["Mismatch rows"].sum() == 0
folder_time_sums_ok = folder_summary_df["Difference"].abs().sum() == 0

if df_out_ok and csv_ok and folder_time_sums_ok and split_total_ok:
    print("✅ PASS: df_out and all generated Nexus CSV files have durations matching their start/end times.")
else:
    print("⚠️ FAIL: At least one duration mismatch exists. Check the summaries above.")

===== FINAL AUDIT: DURATIONS + OUTPUT FOLDER SUMS =====

--- Adjusted df_out ---
Rows checked                 : 3,120
Sum duration column          : 60216
Sum calculated from times    : 60216
Difference                   : 0
Rows with duration mismatch  : 0
✅ OK: duration matches start/end.

===== GENERATED CSV AUDIT =====

--- 01_All_Collabs_One_CSV / RDA_AllCollabs+60216.csv ---
Rows checked                 : 3,120
Sum duration column          : 60216
Sum calculated from times    : 60216
Difference                   : 0
Rows with duration mismatch  : 0
✅ OK: duration matches start/end.

--- 02_Collabs_With_61010_One_CSV / RDA_CollabsWith61010+58882.csv ---
Rows checked                 : 3,091
Sum duration column          : 58882
Sum calculated from times    : 58882
Difference                   : 0
Rows with duration mismatch  : 0
✅ OK: duration matches start/end.

--- 03_Per_Collab_Separate / 1209-Sihyürek Mine+7234.csv ---
Rows checked                 : 360
Sum duration column     

,Folder,Folder exists,CSV files,Rows,Sum Dauer_verrechnet,Sum calculated Von/Bis,Difference,Mismatch rows
0,01_All_Collabs_One_CSV,True,1,3120,60216,60216,0,0
1,02_Collabs_With_61010_One_CSV,True,1,3091,58882,58882,0,0
2,03_Per_Collab_Separate,True,13,3120,60216,60216,0,0
3,02_Whitelisted_Ready_For_101,True,0,0,0,0,0,0



Reference:
df_out total duration: 60216

Expected logic:
01_All_Collabs_One_CSV should usually equal df_out total.
03_Per_Collab_Separate should usually equal df_out total.
02_Collabs_With_61010_One_CSV will be smaller unless every collaborator has 61010.

===== PER CSV FILE SUMMARY =====


,Folder,CSV file,Rows,Sum Dauer_verrechnet,Sum calculated Von/Bis,Difference,Mismatch rows
0,01_All_Collabs_One_CSV,01_All_Collabs_One_CSV/RDA_AllCollabs+60216.csv,3120,60216,60216,0,0
1,02_Collabs_With_61010_One_CSV,02_Collabs_With_61010_One_CSV/RDA_CollabsWith6...,3091,58882,58882,0,0
2,03_Per_Collab_Separate,03_Per_Collab_Separate/RDA-1209-Sihyürek Mine...,360,7234,7234,0,0
3,03_Per_Collab_Separate,03_Per_Collab_Separate/RDA-2004-Incerti Mélan...,253,4792,4792,0,0
4,03_Per_Collab_Separate,03_Per_Collab_Separate/RDA-2008-Le Pavec Anna+...,410,6979,6979,0,0
5,03_Per_Collab_Separate,03_Per_Collab_Separate/RDA-2012-Belbahri Hakim...,335,7732,7732,0,0
6,03_Per_Collab_Separate,03_Per_Collab_Separate/RDA-2014-Mojsilovic Tam...,393,7212,7212,0,0
7,03_Per_Collab_Separate,03_Per_Collab_Separate/RDA-2016-Tissot Katia+6...,11,660,660,0,0
8,03_Per_Collab_Separate,03_Per_Collab_Separate/RDA-2017-Sbarzella Andr...,422,7266,7266,0,0
9,03_Per_Collab_Separate,03_Per_Collab_Separate/RDA-2020-Hafidi Moustap...,99,1836,1836,0,0



✅ All generated Nexus CSV rows have Dauer_verrechnet matching Von/Bis.

===== SPLIT TOTAL CHECK =====
Main raw duration after whitelist removal : 60216
Whitelisted raw duration                 : 0
Raw split total                          : 60216
Adjusted main df_out duration            : 60216
Whitelist transfer CSV duration          : 0
Combined export duration                 : 60216
OK: main adjusted duration + whitelist transfer duration equals the split raw total.

===== FINAL VERDICT =====
✅ PASS: df_out and all generated Nexus CSV files have durations matching their start/end times.


In [ ]:
# @title ✅ FINAL SANITY CHECK — Total Minutes, Mappings, Collabs & Codes

import pandas as pd

print("===== FINAL SANITY CHECK: TOTAL MINUTES =====\n")

# 1. Total Starting Minutes (Raw DataFrame)
if 'df' in globals() and duree_col in df.columns:
    total_start = int(pd.to_numeric(df[duree_col], errors='coerce').fillna(0).sum())
else:
    total_start = 0
    print("⚠️ Could not find the original raw dataframe 'df' or duration column.")

# 2. Total Exported to 201 (Adjusted Main Output)
if 'df_out' in globals() and duree_col in df_out.columns:
    total_201 = int(pd.to_numeric(df_out[duree_col], errors='coerce').fillna(0).sum())
else:
    total_201 = 0
    print("⚠️ Could not find 'df_out' for 201 exports.")

# 3. Total Exported to 101 (Whitelisted Transfer)
if 'whitelist_transfer_csv_df' in globals() and 'Dauer_verrechnet' in whitelist_transfer_csv_df.columns:
    total_101 = int(pd.to_numeric(whitelist_transfer_csv_df['Dauer_verrechnet'], errors='coerce').fillna(0).sum())
else:
    total_101 = 0
    print("⚠️ Could not find 'whitelist_transfer_csv_df' for 101 exports.")

# Calculations
total_combined_exports = total_201 + total_101
difference = total_combined_exports - total_start

print(f"Total Starting Minutes (Raw file)      : {total_start:,}")
print("-" * 50)
print(f"Total Exported to 201 (Main Adjusted)  : {total_201:,}")
print(f"Total Exported to 101 (Whitelisted)    : {total_101:,}")
print(f"Total Combined Exports (201 + 101)     : {total_combined_exports:,}")
print("-" * 50)
print(f"Difference (Combined Exports - Start)  : {difference:,}")

if difference == 0:
    print("\n✅ SUCCESS: The combined exported minutes perfectly match the starting raw minutes!")
else:
    print(f"\n⚠️ WARNING: There is a mismatch of {difference} minutes between the starting file and the combined exports.")


print("\n===== MAPPING & BLANK VALUES CHECK =====")
if 'df_whitelist_mapped' in globals() and 'orig_client_no' in df_whitelist_mapped.columns and 'orig_collab_no' in df_whitelist_mapped.columns:
    mask_unmapped_client = df_whitelist_mapped["orig_client_no"].notna() & (df_whitelist_mapped["orig_client_no"] != 0) & (df_whitelist_mapped["KD-Nr"] == 0)
    mask_unmapped_collab = df_whitelist_mapped["orig_collab_no"].notna() & (df_whitelist_mapped["orig_collab_no"] != 0) & (df_whitelist_mapped["Mitarbeiter-ID"] == 0)
    unmapped_clients = df_whitelist_mapped[mask_unmapped_client]["orig_client_no"].unique().tolist()
    unmapped_collabs = df_whitelist_mapped[mask_unmapped_collab]["orig_collab_no"].unique().tolist()

    if len(unmapped_clients) > 0:
        print(f"⚠️ Unmapped Clients (101 transfer): {unmapped_clients}")
    else:
        print("✅ All clients successfully mapped in 101 transfer.")

    if len(unmapped_collabs) > 0:
        print(f"⚠️ Unmapped Collaborators (101 transfer): {unmapped_collabs}")
    else:
        print("✅ All collaborators successfully mapped in 101 transfer.")
else:
    print("No whitelist mapping performed (101 transfer skipped or variables not found).")

if 'df' in globals():
    client_col_name = next((c for c in ['N° du client', 'No client', 'ID client', 'Client'] if c in df.columns), None)
    if client_col_name:
        blank_clients = df[df[client_col_name].isna() | (df[client_col_name] == 0) | (df[client_col_name] == '') | (df[client_col_name].astype(str).str.strip() == '')]
        if not blank_clients.empty:
            print(f"⚠️ Found {len(blank_clients)} rows with BLANK/ZERO clients in RAW data.")
        else:
            print("✅ No blank/zero clients found in RAW data.")

    collab_col_name = next((c for c in ['No collaborateur', 'Collaborateur', 'ID collaborateur'] if c in df.columns), None)
    if collab_col_name:
        blank_collabs = df[df[collab_col_name].isna() | (df[collab_col_name] == 0) | (df[collab_col_name] == '') | (df[collab_col_name].astype(str).str.strip() == '')]
        if not blank_collabs.empty:
            print(f"⚠️ Found {len(blank_collabs)} rows with BLANK/ZERO collaborators in RAW data.")
        else:
            print("✅ No blank/zero collaborators found in RAW data.")


print("\n===== LIST OF ALL COLLABORATORS =====")
if 'df' in globals() and 'collab_col_name' in locals() and collab_col_name:
    all_collabs = sorted([str(x) for x in df[collab_col_name].dropna().unique()])
    print(f"Total Unique Collaborator IDs in RAW data: {len(all_collabs)}")
    print(", ".join(all_collabs))

    if 'Collaborateur' in df.columns and collab_col_name != 'Collaborateur':
        print("\nCollaborator Names:")
        all_collab_names = sorted([str(x) for x in df['Collaborateur'].dropna().unique()])
        print(" | ".join(all_collab_names))

print("\n===== ALL PRESTATION CODES USED =====")
if 'df' in globals():
    code_col_name = next((c for c in ['N° Prestation', 'No prestation', 'Code prestation', 'Prestation', 'Code'] if c in df.columns), None)
    if code_col_name:
        all_codes = sorted([str(x) for x in df[code_col_name].dropna().unique()])
        print(f"Total Unique Codes in RAW data: {len(all_codes)}")
        print(", ".join(all_codes))


===== FINAL SANITY CHECK: TOTAL MINUTES =====

⚠️ Could not find 'whitelist_transfer_csv_df' for 101 exports.
Total Starting Minutes (Raw file)      : 60,216
--------------------------------------------------
Total Exported to 201 (Main Adjusted)  : 60,216
Total Exported to 101 (Whitelisted)    : 0
Total Combined Exports (201 + 101)     : 60,216
--------------------------------------------------
Difference (Combined Exports - Start)  : 0

✅ SUCCESS: The combined exported minutes perfectly match the starting raw minutes!

===== MAPPING & BLANK VALUES CHECK =====
No whitelist mapping performed (101 transfer skipped or variables not found).
⚠️ Found 1324 rows with BLANK/ZERO clients in RAW data.
✅ No blank/zero collaborators found in RAW data.

===== LIST OF ALL COLLABORATORS =====
Total Unique Collaborator IDs in RAW data: 13
1209, 2004, 2008, 2012, 2014, 2016, 2017, 2020, 2023, 2026, 2027, 2029, 30002

Collaborator Names:
Belbahri Hakima | Hafidi Moustapha | Incerti Mélanie | Le Pavec A

In [ ]:
# @title Details of Unmapped Collaborators and Clients in 101 Transfer
import pandas as pd
from IPython.display import display

if 'df_whitelist_mapped' in globals() and 'orig_collab_no' in df_whitelist_mapped.columns and 'orig_client_no' in df_whitelist_mapped.columns:
    # Mask for unmapped collaborators
    mask_unmapped_collab = (df_whitelist_mapped["orig_collab_no"].notna() &
                            (df_whitelist_mapped["orig_collab_no"] != 0) &
                            (df_whitelist_mapped["Mitarbeiter-ID"] == 0))
    unmapped_collabs_df = df_whitelist_mapped[mask_unmapped_collab]

    print("\u26a0\ufe0f UNMAPPED COLLABORATORS \u26a0\ufe0f")
    if not unmapped_collabs_df.empty:
        collab_id_col = next((c for c in ['No collaborateur', 'ID collaborateur'] if c in unmapped_collabs_df.columns), 'orig_collab_no')
        collab_name_col = next((c for c in ['Collaborateur', 'Nom Collaborateur'] if c in unmapped_collabs_df.columns), None)

        cols_to_show = [collab_id_col]
        if collab_name_col:
            cols_to_show.append(collab_name_col)

        display(unmapped_collabs_df[cols_to_show].drop_duplicates())
    else:
        print("\u2705 All collaborators mapped successfully.")

    print("\n\u26a0\ufe0f UNMAPPED CLIENTS \u26a0\ufe0f")
    mask_unmapped_client = (df_whitelist_mapped["orig_client_no"].notna() &
                            (df_whitelist_mapped["orig_client_no"] != 0) &
                            (df_whitelist_mapped["KD-Nr"] == 0))
    unmapped_clients_df = df_whitelist_mapped[mask_unmapped_client]

    if not unmapped_clients_df.empty:
        client_id_col = next((c for c in ['N\u00b0 du client', 'No client', 'ID client'] if c in unmapped_clients_df.columns), 'orig_client_no')
        cols_to_show_client = [client_id_col]
        if 'Client' in unmapped_clients_df.columns and client_id_col != 'Client':
            cols_to_show_client.append('Client')

        display(unmapped_clients_df[cols_to_show_client].drop_duplicates())
    else:
        print("\u2705 All clients mapped successfully.")
else:
    print("Data 'df_whitelist_mapped' not found or mapping was skipped (no whitelist data). Please run the whitelist transfer cell first if needed.")

Data 'df_whitelist_mapped' not found or mapping was skipped (no whitelist data). Please run the whitelist transfer cell first if needed.


In [ ]:
# @title 🔍 Verification: Why is the 61010 collab total 190,813 instead of 193,856?

# Get the list of collaborators who have at least one 61010
collabs_with_61010_list = df_out[df_out['No prestation'].astype(str).str.strip() == '61010']['No collaborateur'].unique()

# 1. Duration for these collabs in the Main Export (201)
duration_in_main = df_out[df_out['No collaborateur'].isin(collabs_with_61010_list)]['Durée'].sum()

# 2. Duration for these SAME collabs in the Whitelisted Transfer (101)
duration_in_whitelist = df_whitelist[df_whitelist['No collaborateur'].isin(collabs_with_61010_list)]['Durée'].sum()

print("===== RECONCILING 61010 COLLABORATORS' TOTAL DURATION =====\n")
print(f"Duration in Main 201 Export (without whitelisted codes) : {duration_in_main:,.0f} mins")
print(f"Duration in Whitelisted 101 Export (for these collabs)  : {duration_in_whitelist:,.0f} mins")
print("-" * 65)
print(f"Total RAW Duration for these collabs (Your external check): {duration_in_main + duration_in_whitelist:,.0f} mins")


===== RECONCILING 61010 COLLABORATORS' TOTAL DURATION =====

Duration in Main 201 Export (without whitelisted codes) : 58,882 mins
Duration in Whitelisted 101 Export (for these collabs)  : 0 mins
-----------------------------------------------------------------
Total RAW Duration for these collabs (Your external check): 58,882 mins
